# M26 · Chaînes de Markov et équation de Bellman — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m26_markov_bellman/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). Dans les parties A à D, donne les valeurs que rend le calcul, **sans les arrondir** : les décimales demandées sont celles de ta copie. « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m26_markov_bellman/03_notebook.ipynb`, créée par `python tools/start_chapter.py M26`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M26.1 à M26.9, M26.14 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M26.16 | La loi d'une chaîne après n pas, en convention ligne · `n_step_distribution` | 🔨 | ★ | 20 |
|  | M26.17 | La loi stationnaire par un système linéaire, et le refus d'une chaîne qui en a plusieurs · `stationary_distribution` | 🔨 | ★★ | 25 |
|  | M26.18 | Le système de Bellman d'une politique fixée, avec une fin possible (matrice sous-stochastique) et γ = 1 quand la fin est sûre · `bellman_linear_solve` | 🔨 | ★★ | 25 |
|  | M26.19 | Les itérés de l'opérateur de Bellman d'une politique fixée (Richardson de pas 1) · `bellman_fixed_point_iterates` | 🔨 | ★ | 20 |
|  | M26.20 | La loi stationnaire de trois façons, `np.linalg.eig` sur Pᵀ, `scipy.linalg.null_space` et la méthode de la puissance, sur le surfeur aléatoire d'un web de six pages | 📦 | ★ | 20 |
| B | M26.21 | Prédire la somme des lignes de (1 − γ)(I − γP)⁻¹, puis leur limite quand γ tend vers 1, et vérifier | 🔮 | ★ | 15 |
|  | M26.22 | Le biais du maximum par simulation : k fournisseurs de même qualité, n mesures chacun, le meilleur mesuré contre l'estimateur double | 🔬 | ★★ | 20 |
|  | M26.23 | La loi d'une chaîne au fil des pas depuis trois départs, et ‖μ_n − π‖₁ en échelle logarithmique contre \|λ₂\|ⁿ | 🎨 | ★ | 15 |
| C | M26.24 | Chasse à l'erreur : une matrice lue dans la mauvaise convention, une loi stationnaire non normalisée, un état final qui rapporte encore, γ = 1 sans fin | 🐛 | ★ | 25 |
|  | M26.25 | Un test de propriété : πP = π, et pour `bellman_fixed_point_iterates`, V_k − V tenu par γᵏ‖V₀ − V‖∞ et V = r + γPV pour la limite, sur des chaînes tirées au hasard | 🛠️ | ★ | 20 |
| D | M26.26 | Défi : estimer la valeur d'un état à 0,01 près en simulant l'arrêt géométrique (continuer avec la probabilité γ) et en sommant les récompenses sans actualisation, sous un budget de pas simulés | 🏆 | ★★★ | 30 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M26")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : une loi à n pas et un coût moyen, une loi stationnaire et trois chaînes qui posent problème, l'analyse au premier pas, une durée géométrique, deux systèmes de Bellman et la valeur d'une action, leurs itérés, un arrêt optimal, le biais du maximum, et un pont vers les tableaux à trois axes.
- Écrire les quatre fonctions de `mymath/markov.py` (`n_step_distribution`, `stationary_distribution`, `bellman_linear_solve`, `bellman_fixed_point_iterates`) et les tester.
- Calculer un PageRank de trois façons, prédire la loi d'occupation actualisée, simuler le biais du maximum, dessiner l'oubli du départ, corriger quatre erreurs classiques, écrire un test de propriété, et relever un défi.

**Rappel express.** Une loi est une ligne : $\boldsymbol{\mu}_{n+1} = \boldsymbol{\mu}_n\mathbf{P}$, $\boldsymbol{\mu}_n = \boldsymbol{\mu}_0\mathbf{P}^n$ ; une fonction est une colonne : $\mathbb{E}[f(S_1) \mid S_0 = i] = (\mathbf{P}\mathbf{f})_i$. Loi stationnaire : $\boldsymbol{\pi}\mathbf{P} = \boldsymbol{\pi}$, $\sum_j\pi_j = 1$ (le système $(\mathbf{P}^\top - \mathbf{I})\boldsymbol{\pi}^\top = \mathbf{0}$, sa dernière équation remplacée par la somme) ; écart en $\lvert\lambda_2\rvert^n$. Premier pas : $\mathbf{h} = \mathbf{Q}\mathbf{h} + \mathbf{b}$, $\mathbf{t} = \mathbf{1} + \mathbf{Q}\mathbf{t}$. Bellman : $(\mathbf{I} - \gamma\mathbf{P})\mathbf{V} = \mathbf{r}$, inversible pour $\gamma < 1$, et pour $\gamma = 1$ si la fin est sûre ; $(1 - \gamma)(\mathbf{I} - \gamma\mathbf{P})^{-1}$ est stochastique. Itérés : $\mathbf{V}_{k+1} = \mathbf{r} + \gamma\mathbf{P}\mathbf{V}_k$, $\lVert\mathbf{V}_k - \mathbf{V}\rVert_\infty \le \gamma^k\lVert\mathbf{V}_0 - \mathbf{V}\rVert_\infty$ et $\le \frac{\gamma}{1 - \gamma}\lVert\mathbf{V}_k - \mathbf{V}_{k-1}\rVert_\infty$. Arrêt géométrique : continuer avec la probabilité $\gamma$ et sommer sans actualiser donne $V$ en moyenne. Biais du maximum : $\mathbb{E}[\max_i\hat{X}_i] \ge \max_i\mathbb{E}[\hat{X}_i]$ ; l'estimateur double choisit sur un échantillon et mesure sur l'autre.

## Partie 0 · Vérifier tes exercices papier (✏️ M26.1 à M26.9 ; 🔗 M26.14)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question (« exact values » : tous les chiffres, ces calculs tombent juste) ;
- une liste `[0.2, -1]` ; une matrice comme la liste de ses **lignes**, `[[1, 0], [-2, 3]]` ;
- une fraction irréductible entre guillemets, `"3/7"`, quand la question le dit ;
- une **forme** comme un tuple, `(7, 2)`, ou comme un texte, `"(7, 2)"` ;
- `True` ou `False` pour un vrai ou faux.

Les réponses pas encore remplies affichent ⏳.

**Ex M26.1 — Des vélos en libre-service entre trois stations : loi après deux et trois jours, et coût moyen du lendemain**

In [ ]:
# Ex M26.1: the values you found on paper
answer_M26_1a = ...  # a) a number
answer_M26_1b = ...  # b) a list of 3 numbers (exact values)
answer_M26_1c = ...  # c) a list of 3 numbers (exact values)
answer_M26_1d = ...  # d) a number
answer_M26_1e = ...  # e) a list of 3 numbers
answer_M26_1f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_M26_1a, answer_M26_1b, answer_M26_1c, answer_M26_1d, answer_M26_1e, answer_M26_1f]):
    wbm.check(f"M26.1{letter}", answer)

**Ex M26.2 — Trois opérateurs et leurs parts de marché : la loi stationnaire par un système linéaire**

In [ ]:
# Ex M26.2: the values you found on paper
answer_M26_2a = ...  # a) a list of 3 numbers (4 decimals)
answer_M26_2b = ...  # b) an irreducible fraction, as text "p/q"
answer_M26_2c = ...  # c) a list of 3 numbers (exact values)
answer_M26_2d = ...  # d) an irreducible fraction, as text "p/q"
answer_M26_2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_M26_2a, answer_M26_2b, answer_M26_2c, answer_M26_2d, answer_M26_2e]):
    wbm.check(f"M26.2{letter}", answer)

**Ex M26.3 — Trois chaînes qui posent problème : un bac, un site bilingue, une période d'essai**

In [ ]:
# Ex M26.3: the values you found on paper
answer_M26_3a = ...  # a) a list of 4 numbers (4 decimals)
answer_M26_3b = ...  # b) a list of 4 numbers (4 decimals)
answer_M26_3c = ...  # c) True or False
answer_M26_3d = ...  # d) a list of 4 numbers (4 decimals)
answer_M26_3e = ...  # e) True or False
answer_M26_3f = ...  # f) a list of 3 numbers (exact values)

for letter, answer in zip("abcdef", [answer_M26_3a, answer_M26_3b, answer_M26_3c, answer_M26_3d, answer_M26_3e, answer_M26_3f]):
    wbm.check(f"M26.3{letter}", answer)

**Ex M26.4 — Un cursus en trois années : la probabilité d'être diplômé et la durée moyenne des études**

In [ ]:
# Ex M26.4: the values you found on paper
answer_M26_4a = ...  # a) a list of 3 numbers (3 decimals)
answer_M26_4b = ...  # b) an irreducible fraction, as text "p/q"
answer_M26_4c = ...  # c) a list of 3 numbers (2 decimals)
answer_M26_4d = ...  # d) a number (2 decimals)
answer_M26_4e = ...  # e) a number (2 decimals)

for letter, answer in zip("abcde", [answer_M26_4a, answer_M26_4b, answer_M26_4c, answer_M26_4d, answer_M26_4e]):
    wbm.check(f"M26.4{letter}", answer)

**Ex M26.5 — Un abonné qui peut résilier chaque mois : durée géométrique, revenu total moyen et revenu actualisé**

In [ ]:
# Ex M26.5: the values you found on paper
answer_M26_5a = ...  # a) a number (4 decimals)
answer_M26_5b = ...  # b) an irreducible fraction, as text "p/q"
answer_M26_5c = ...  # c) a number
answer_M26_5d = ...  # d) a number
answer_M26_5e = ...  # e) a number (3 decimals)
answer_M26_5f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_M26_5a, answer_M26_5b, answer_M26_5c, answer_M26_5d, answer_M26_5e, answer_M26_5f]):
    wbm.check(f"M26.5{letter}", answer)

**Ex M26.6 — Une machine qui s'use, deux politiques d'entretien : deux systèmes de Bellman et la valeur d'une action**

In [ ]:
# Ex M26.6: the values you found on paper
answer_M26_6a = ...  # a) a number
answer_M26_6b = ...  # b) a 3 × 3 matrix
answer_M26_6c = ...  # c) a list of 3 numbers (2 decimals)
answer_M26_6d = ...  # d) a list of 3 numbers (2 decimals)
answer_M26_6e = ...  # e) a number (2 decimals)
answer_M26_6f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_M26_6a, answer_M26_6b, answer_M26_6c, answer_M26_6d, answer_M26_6e, answer_M26_6f]):
    wbm.check(f"M26.6{letter}", answer)

**Ex M26.7 — Les itérés de Bellman de la machine : synchrones, en place, et le nombre d'itérations**

In [ ]:
# Ex M26.7: the values you found on paper
answer_M26_7a = ...  # a) a list of 3 numbers (exact values)
answer_M26_7b = ...  # b) a number (4 decimals)
answer_M26_7c = ...  # c) a list of 3 numbers (exact values)
answer_M26_7d = ...  # d) a number (4 decimals)
answer_M26_7e = ...  # e) an integer
answer_M26_7f = ...  # f) an irreducible fraction, as text "p/q"
answer_M26_7g = ...  # g) a number (2 decimals)

for letter, answer in zip("abcdefg", [answer_M26_7a, answer_M26_7b, answer_M26_7c, answer_M26_7d, answer_M26_7e, answer_M26_7f, answer_M26_7g]):
    wbm.check(f"M26.7{letter}", answer)

**Ex M26.8 — Vendre une voiture d'occasion : valeur de continuation et règle à seuil**

In [ ]:
# Ex M26.8: the values you found on paper
answer_M26_8a = ...  # a) a number
answer_M26_8b = ...  # b) a number (rounded to the euro)
answer_M26_8c = ...  # c) a list of integers
answer_M26_8d = ...  # d) a number (rounded to the euro)
answer_M26_8e = ...  # e) a number (rounded to the euro)
answer_M26_8f = ...  # f) a number (rounded to the euro)

for letter, answer in zip("abcdef", [answer_M26_8a, answer_M26_8b, answer_M26_8c, answer_M26_8d, answer_M26_8e, answer_M26_8f]):
    wbm.check(f"M26.8{letter}", answer)

**Ex M26.9 — Le meilleur de deux fournisseurs sur des mesures bruitées : le biais du maximum**

In [ ]:
# Ex M26.9: the values you found on paper
answer_M26_9a = ...  # a) a list of 2 numbers
answer_M26_9b = ...  # b) a number (2 decimals)
answer_M26_9c = ...  # c) a number (3 decimals)
answer_M26_9d = ...  # d) a number (2 decimals)
answer_M26_9e = ...  # e) True or False
answer_M26_9f = ...  # f) a number (2 decimals)

for letter, answer in zip("abcdef", [answer_M26_9a, answer_M26_9b, answer_M26_9c, answer_M26_9d, answer_M26_9e, answer_M26_9f]):
    wbm.check(f"M26.9{letter}", answer)

**Ex M26.14 — Pont : d'une politique stochastique à la chaîne qu'elle induit, avec des tableaux à trois axes**

In [ ]:
# Ex M26.14: the values you found on paper
answer_M26_14a = ...  # a) a shape: a tuple (7, 2) or a text "(7, 2)"
answer_M26_14b = ...  # b) a 2 × 3 matrix
answer_M26_14c = ...  # c) a list of 3 numbers
answer_M26_14d = ...  # d) a list of 3 numbers
answer_M26_14e = ...  # e) a list of 3 numbers (2 decimals)
answer_M26_14f = ...  # f) a number (2 decimals)
answer_M26_14g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M26_14a, answer_M26_14b, answer_M26_14c, answer_M26_14d, answer_M26_14e, answer_M26_14f, answer_M26_14g]):
    wbm.check(f"M26.14{letter}", answer)

## Partie A · Ta librairie : la loi à n pas, la loi stationnaire, le système de Bellman et ses itérés ; puis NumPy et SciPy

Fiche §1, §2, §5 et §6. Les quatre fonctions de `mymath/markov.py`, leurs tests, et un usage de chacune sur les chaînes des exercices papier ; puis trois façons de calculer un PageRank. La cellule de données ci-dessous recopie les données des exercices papier (la machine par ses lignes et ses récompenses : c'est à toi de construire $\mathbf{P}_\pi$ et $\mathbf{r}_\pi$).

In [ ]:
# Tools of the notebook (parts A to D)
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from scipy import linalg

from wbm.errors import MymathMissingError
from wbm.nbtools import StopCheck, checked, drawn, filled, fr, own_code, pair, run_pytest_group, same, verdict

TEST_M26 = "tests/test_m26_markov.py"              # the tests of mymath/markov.py
REF = wbm.load_mymath("ref", alias="mymath_ref")   # the reference: the data cells and the checks never use your code
ref_markov = REF.markov
REF_MARKOV = {f"ref_markov.{name}": getattr(ref_markov, name)    # a check stops if your code calls one of them
              for name in ("n_step_distribution", "stationary_distribution", "bellman_linear_solve",
                           "bellman_fixed_point_iterates")}


def shown(value, digits=4):
    """A short text for a result of the learner's code (the check messages): numbers rounded, as lists."""
    if value is None:
        return "None"
    try:
        if isinstance(value, tuple):
            return "(" + ", ".join(str(np.round(np.asarray(v, dtype=float), digits).tolist()) for v in value) + ")"
        return str(np.round(np.asarray(value, dtype=float), digits).tolist())
    except (TypeError, ValueError):
        return repr(value)[:120]


def check_answer(ex_id, value, hint=None):
    """wbm.check(ex_id, value, computed=True). For a list that holds True or False, the message gives only how many
    elements are right, not the place of the first wrong one (it would let you guess the booleans one by one).
    hint(value), when given, may return a targeted « Piste » for a wrong answer (None otherwise)."""
    holds_bool = isinstance(value, (list, tuple)) and any(isinstance(v, (bool, np.bool_)) for v in value)
    if not holds_bool and hint is None:
        return wbm.check(ex_id, value, computed=True)
    result = wbm.check(ex_id, value, computed=True, quiet=True)
    message = result.message
    if holds_bool:
        for cut in (" Le premier faux est le ", " Premier élément faux à l'indice "):
            message = message.split(cut)[0]
    if hint is not None and result.status not in ("correct", "pending", "unknown"):
        try:
            extra = hint(value)
        except Exception:  # noqa: BLE001 - an odd answer: no targeted hint
            extra = None
        if extra:
            message = f"{message} Piste : {extra}"
    icon = {"correct": "✅", "pending": "⏳", "unknown": "❓"}.get(result.status, "❌")
    print(f"{icon} Ex {ex_id} : {message}")
    return result


def reporter(results, stop=True):
    """A function report(ok, success, failure) that prints one ✅ or ❌ line, keeps the verdict in results, and (stop)
    ends the check at the first ❌ (StopCheck, caught by checked)."""
    def report(ok, success, failure):
        results.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))
        if not ok and stop:
            raise StopCheck
    return report


def matches(got, want, rtol=1e-9, atol=1e-12):
    """True when got is an array of numbers of the shape of want and equal to it up to rounding (no error for an odd
    got): to recognize a classic wrong result before the comparison by same."""
    try:
        arr = np.asarray(got, dtype=float)
    except (TypeError, ValueError):
        return False
    want = np.asarray(want, dtype=float)
    return arr.shape == want.shape and bool(np.allclose(arr, want, rtol=rtol, atol=atol))


def line_points(line):
    """The (x, y) points of a line of matplotlib, as two float arrays."""
    return np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)


def through(ax, xs, ys, rtol=1e-7, atol=1e-12, dashed=None):
    """True when one line of ax passes by all the points (xs, ys), in this order (dashed=True: a dashed or dotted line
    only; False: a solid one only)."""
    xs, ys = np.asarray(xs, dtype=float), np.asarray(ys, dtype=float)
    for line in ax.lines:
        if dashed is not None and (line.get_linestyle() not in ("-", "solid")) != dashed:
            continue
        x, y = line_points(line)
        if (x.shape == xs.shape and y.shape == ys.shape and np.allclose(x, xs, rtol=rtol, atol=atol)
                and np.allclose(y, ys, rtol=rtol, atol=atol)):
            return True
    return False


def horizontal_at(ax, level, rtol=1e-7):
    """True when a line of ax is horizontal at the height level (ax.axhline, or a line of constant y)."""
    return any(len(line_points(line)[1]) >= 2 and np.allclose(line_points(line)[1], level, rtol=rtol, atol=1e-12)
               for line in ax.lines)


# The data of the paper exercises (02_exercices.md), the states numbered in the order of each statement
P_BIKES = np.array([[0.6, 0.3, 0.1], [0.3, 0.4, 0.3], [0.4, 0.2, 0.4]])   # M26.1: the stations A, B, C
MU0_BIKES = np.array([0.4, 0.4, 0.2])                      # 80, 80 and 40 bikes out of 200
COST_BIKES = np.array([0.4, 0.8, 2.0])                     # the recharge of a bike in A, B, C (€): f
P_OPERATORS = np.array([[0.9, 0.05, 0.05], [0.05, 0.85, 0.1], [0.2, 0.15, 0.65]])   # M26.2: Azur, Boréal, Cobalt
SHARES_OPERATORS = np.array([0.45, 0.4, 0.15])             # today's market shares
P_FERRY = np.array([[0, 0, 0.7, 0.3], [0, 0, 0.35, 0.65], [0.9, 0.1, 0, 0], [0.3, 0.7, 0, 0]])   # M26.3: A, B, C, D
P_SITE = np.array([[0.8, 0.2, 0, 0], [0.35, 0.65, 0, 0], [0, 0, 0.55, 0.45], [0, 0, 0.3, 0.7]])  # FR 0, 1; EN 2, 3
MU0_SITE = np.array([0.6, 0, 0.4, 0])                      # where the visitors arrive
P_TRIAL = np.array([[0.4, 0.45, 0.15], [0, 0.9, 0.1], [0, 0.4, 0.6]])     # trial, regular, occasional
Q_CURSUS = np.array([[0.15, 0.7, 0], [0.05, 0.15, 0.75], [0, 0, 0.15]])   # M26.4: L1, L2, L3 among themselves
GRADUATE_CURSUS = np.array([0, 0, 0.75])                   # b, the column « diplômé »
DROPOUT_CURSUS = np.array([0.15, 0.05, 0.1])               # the column « abandon »
FEES_CURSUS = np.array([170.0, 170.0, 240.0])              # € per year in L1, L2, L3
GAMMA_MACHINE = 0.6                                        # M26.6: the machine, new (0), worn (1), broken (2), in k€
ROW_NEW = np.array([0.7, 0.3, 0])                          # new, « continuer »: it brings 10
ROW_WORN_CONTINUE = np.array([0, 0.65, 0.35])              # worn, « continuer »: 6 if it stays worn, 3 if it breaks
ROW_WORN_MAINTAIN = np.array([0.95, 0, 0.05])              # worn, « entretenir »: it brings 6 - 4 = 2
ROW_BROKEN = np.array([0.8, 0, 0.2])                       # broken, « réparer »: it brings -8

### Ex M26.16 — La loi d'une chaîne après n pas, en convention ligne · `n_step_distribution` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer la loi après n pas μ₀Pⁿ en convention ligne, puis voir une loi se fixer en quelques pas et une autre alterner sans fin.  
**Prérequis :** fiche §1 ; M03, M12c ; Ex M26.1 et M26.3 · **mymath :** `markov.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M25b). Ouvre ta copie `mon_travail/mymath/markov.py`, lis la docstring du module (ses conventions, et ce qui est permis) puis celle de la fonction, remplace chaque `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Dans le notebook, appelle tes fonctions par `mymath.markov.<nom>` (par exemple `mymath.markov.n_step_distribution(P, mu0, n)`), ou donne-leur un nom court (`n_step_distribution = mymath.markov.n_step_distribution`) dans une cellule exécutée après le rechargement. ⚠️ Le squelette n'importe rien : ajoute en tête du fichier `import numpy as np` (et `import numbers`, `import math` si tu t'en sers). NumPy est permis, mais de `np.linalg` seulement `solve` et `matrix_rank`, et seulement dans `stationary_distribution` et `bellman_linear_solve` ; ni `np.linalg.matrix_power`, `eig`, `eigvals`, `inv`, ni SciPy : les tests comparent tes résultats aux leurs, et un test lit ton module. Une loi est une **ligne** (`mu @ P`), une valeur une **colonne** (`P @ V`). Rends des tableaux NumPy de flottants, et ne modifie jamais les tableaux reçus. Hors de `mymath`, le notebook peut se servir de tout NumPy et de SciPy.

Écris `n_step_distribution(P, mu0, n)` (fiche §1), en suivant sa docstring : contrôle `P` (carrée, finie, coefficients $\ge 0$, lignes de somme 1 à $10^{-9}$ près), `mu0` (une loi à $m$ coordonnées, de somme 1 à $10^{-9}$ près) et `n` (un entier $\ge 0$ : un entier NumPy convient, mais ni un booléen ni un flottant comme `2.0`), puis fais $n$ produits `mu @ P` (ou l'élévation au carré que suggère le TODO) ; pour $n = 0$, rends une copie flottante de `mu0`. `ValueError` pour ce que refuse la docstring.

**L'usage.** La cellule de données de la partie A donne les vélos de M26.1 (`P_BIKES`, `MU0_BIKES`) et le bac de M26.3 (`P_FERRY`). Avec ta fonction :
- a) les lois des vélos dans deux et trois jours (une liste de 2 listes de 3 nombres), à comparer à tes réponses b) et c) de M26.1 ;
- b) pour le bac, depuis A : $\boldsymbol{\mu}_{100}$ et $\boldsymbol{\mu}_{101}$, et vrai ou faux, $\boldsymbol{\mu}_{100}$ égale à $10^{-12}$ près ta réponse b) de M26.3, recalculée exactement (en fractions, pas arrondie à 4 décimales) (une liste de 9 : les 4 nombres de $\boldsymbol{\mu}_{100}$, les 4 de $\boldsymbol{\mu}_{101}$, puis le booléen) ;
- c) vrai ou faux : pour les vélos, `n_step_distribution(P, n_step_distribution(P, mu0, 4), 6)` et `n_step_distribution(P, mu0, 10)` coïncident à $10^{-12}$ près (Chapman-Kolmogorov).

Ce qu'il faut observer : la loi des vélos se fixe en quelques jours ; celle du bac alterne sans fin entre les deux rives.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M26")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M26, "test_n_step_distribution_ or test_module_uses_")

**L'usage** : les vélos, le bac, et Chapman-Kolmogorov.

In [ ]:
answer_M26_16a = answer_M26_16b = answer_M26_16c = ...
with wbm.attempt("M26.16"):          # if you call n_step_distribution here: ⏳ while it is not written
    answer_M26_16a = ...  # a) a list of 2 lists of 3 numbers, as computed: the laws of the bikes in two, then three days
    answer_M26_16b = ...  # b) a list of 9: μ_100 then μ_101 of the ferry from A (8 numbers, as computed), then True or False: μ_100 equals your answer b) of M26.3, recomputed exactly, to 1e-12
    answer_M26_16c = ...  # c) True or False: 4 steps then 6 steps give the law after 10 steps, to 1e-12 (the bikes)

In [ ]:
for letter, answer in zip("abc", [answer_M26_16a, answer_M26_16b, answer_M26_16c]):
    check_answer(f"M26.16{letter}", answer)

### Ex M26.17 — La loi stationnaire par un système linéaire, et le refus d'une chaîne qui en a plusieurs · `stationary_distribution` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer la loi stationnaire par le système carré de la fiche, et voir qu'il détecte l'unicité, pas la période.  
**Prérequis :** fiche §2 ; M09a ; Ex M26.2 et M26.3 · **mymath :** `markov.py` · **Parcours :** C, E

> Même mode d'emploi qu'en M26.16 : écris la fonction dans ta copie `mon_travail/mymath/markov.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `stationary_distribution(P)` (fiche §2), en suivant sa docstring : `A = P.T - I`, dont la dernière ligne est remplacée par des 1, et `b = (0, …, 0, 1)` ; lève une `ValueError` si `np.linalg.matrix_rank(A) < m` ; sinon `np.linalg.solve(A, b)`, les coordonnées négatives dues aux arrondis mises à 0, puis la division par la somme.

**L'usage.** Avec ta fonction et la cellule de données de la partie A :
- a) la loi stationnaire des trois opérateurs de M26.2 (`P_OPERATORS`) (une liste de 3 nombres), à comparer à ta réponse a) ;
- b) celle du bac de M26.3 (`P_FERRY`) (une liste de 4 nombres), à comparer à ta réponse a) : la chaîne est périodique, mais sa loi stationnaire est unique, et ta fonction l'accepte ;
- c) vrai ou faux : ta fonction refuse le site bilingue de M26.3 (`P_SITE`) par une `ValueError` (attrape l'erreur par `try` / `except ValueError`) ;
- d) la loi de la période d'essai de M26.3 (`P_TRIAL`), et vrai ou faux, sa coordonnée 0 est inférieure à $10^{-12}$ (une liste : les 3 nombres, puis le booléen).

Ce qu'il faut observer : le système voit l'unicité, pas la période ; un état transitoire reçoit 0.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M26")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M26, "test_stationary_distribution_")

**L'usage** : les opérateurs, le bac, le site bilingue et la période d'essai.

In [ ]:
answer_M26_17a = answer_M26_17b = answer_M26_17c = answer_M26_17d = ...
with wbm.attempt("M26.17"):          # if you call stationary_distribution here: ⏳ while it is not written
    answer_M26_17a = ...  # a) a list of 3 numbers, as computed: the stationary law of the operators
    answer_M26_17b = ...  # b) a list of 4 numbers, as computed: the stationary law of the ferry
    answer_M26_17c = ...  # c) True or False: your function refuses P_SITE with a ValueError
    answer_M26_17d = ...  # d) a list of 4: the law of the trial period (3 numbers, as computed), then True or False: its coordinate 0 is below 1e-12

In [ ]:
for letter, answer in zip("abcd", [answer_M26_17a, answer_M26_17b, answer_M26_17c, answer_M26_17d]):
    check_answer(f"M26.17{letter}", answer)

### Ex M26.18 — Le système de Bellman d'une politique fixée, avec une fin possible (matrice sous-stochastique) et γ = 1 quand la fin est sûre · `bellman_linear_solve` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer la résolution du système de Bellman d'une politique fixée, et voir qu'une ligne qui finit joue le rôle de γ.  
**Prérequis :** fiche §3 et §5 ; M09a ; Ex M26.4, M26.5 et M26.6 · **mymath :** `markov.py` · **Parcours :** C, E

> Même mode d'emploi qu'en M26.16 : écris la fonction dans ta copie `mon_travail/mymath/markov.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `bellman_linear_solve(P, r, gamma)` (fiche §5), en suivant sa docstring : la solution de $(\mathbf{I} - \gamma\mathbf{P})\mathbf{V} = \mathbf{r}$. Contrôle `P` (carrée, finie, coefficients $\ge 0$, lignes de somme au plus $1 + 10^{-9}$), `r` (1-D, $m$ coordonnées) et `gamma` (un nombre de $[0, 1]$, pas un booléen) ; pour `gamma == 1`, lève une `ValueError` si `np.linalg.matrix_rank(I - P) < m` ; puis `np.linalg.solve`.

**L'usage.** Avec ta fonction et la cellule de données de la partie A :
- a) les valeurs des deux politiques de la machine de M26.6, $\mathbf{V}^{\pi_A}$ puis $\mathbf{V}^{\pi_B}$ (une liste de 2 listes de 3 nombres), à comparer à tes réponses c) et d) : construis toi-même $\mathbf{P}_{\pi_A}$, $\mathbf{r}_{\pi_A}$, $\mathbf{P}_{\pi_B}$ et $\mathbf{r}_{\pi_B}$ à partir des lignes `ROW_…` et des récompenses de l'énoncé ;
- b) le cursus de M26.4 avec $\gamma = 1$ et $\mathbf{P} = \mathbf{Q}$ (`Q_CURSUS`, les trois années, aux lignes sous-stochastiques) : les durées moyennes ($\mathbf{r} = \mathbf{1}$), puis les probabilités d'être diplômé ($\mathbf{r} = \mathbf{b}$, `GRADUATE_CURSUS`) (une liste de 2 listes de 3 nombres), à comparer à tes réponses c) et a) ;
- c) le cursus avec ses deux états absorbants gardés comme états (une matrice $5 \times 5$, dans l'ordre L1, L2, L3, diplômé, abandon, que tu construis avec `Q_CURSUS`, `GRADUATE_CURSUS` et `DROPOUT_CURSUS`, et les frais $(170, 170, 240, 0, 0)$) : vrai ou faux, ta fonction le refuse pour $\gamma = 1$ ; avec $\gamma = 0{,}96$, les frais actualisés depuis L1 ; et vrai ou faux, la matrice $3 \times 3$ $\mathbf{Q}$ avec les frais $(170, 170, 240)$ donne les mêmes trois premières valeurs à $10^{-12}$ près (une liste de 3 : le booléen, le nombre, puis le booléen) ;
- d) l'abonné de M26.5 de deux façons, une ligne qui finit avec $\gamma = 1$ (`P = [[0.7]]`), puis un état qui ne finit jamais avec $\gamma = 0{,}7$ (`P = [[1.0]]`), avec `r = [9]` : les deux valeurs (une liste de 2 nombres), à comparer à tes réponses c) et d).

Ce qu'il faut observer : $\gamma\mathbf{P}$ et une ligne qui finit jouent le même rôle ; avec $\gamma = 1$, un état absorbant gardé comme état rend le système singulier.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M26")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M26, "test_bellman_linear_solve_")

**L'usage** : la machine, le cursus de deux façons, et l'abonné de deux façons.

In [ ]:
answer_M26_18a = answer_M26_18b = answer_M26_18c = answer_M26_18d = ...
with wbm.attempt("M26.18"):          # if you call bellman_linear_solve here: ⏳ while it is not written
    answer_M26_18a = ...  # a) a list of 2 lists of 3 numbers, as computed: V^{π_A}, then V^{π_B} (the machine, γ = 0.6)
    answer_M26_18b = ...  # b) a list of 2 lists of 3 numbers, as computed: the mean durations (r = 1), then the probabilities of graduating (r = b), with P = Q_CURSUS and γ = 1
    answer_M26_18c = ...  # c) a list of 3: True or False: the 5 × 5 matrix is refused for γ = 1; the discounted fees from L1 for γ = 0.96 (as computed); True or False: Q with (170, 170, 240) gives the same three values to 1e-12
    answer_M26_18d = ...  # d) a list of 2 numbers, as computed: P = [[0.7]] with γ = 1, then P = [[1.0]] with γ = 0.7 (r = [9])

In [ ]:
for letter, answer in zip("abcd", [answer_M26_18a, answer_M26_18b, answer_M26_18c, answer_M26_18d]):
    check_answer(f"M26.18{letter}", answer)

### Ex M26.19 — Les itérés de l'opérateur de Bellman d'une politique fixée (Richardson de pas 1) · `bellman_fixed_point_iterates` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer les itérés synchrones de l'opérateur de Bellman, puis compter les itérations qu'il faut vraiment et comparer aux balayages en place.  
**Prérequis :** fiche §6 ; M19, M22 ; Ex M26.6 et M26.7 · **mymath :** `markov.py` · **Parcours :** C, E

> Même mode d'emploi qu'en M26.16 : écris la fonction dans ta copie `mon_travail/mymath/markov.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `bellman_fixed_point_iterates(P, r, gamma, V0=None, n_steps=10)` (fiche §6), en suivant sa docstring : les itérés synchrones $\mathbf{V}_{k+1} = \mathbf{r} + \gamma\mathbf{P}\mathbf{V}_k$, un par ligne d'un tableau de forme $(N + 1, m)$ dont la ligne 0 est `V0` (le vecteur nul s'il manque) ; mêmes contrôles que `bellman_linear_solve`, sans le cas singulier, plus `V0` (1-D, $m$ coordonnées) et `n_steps` (un entier $\ge 0$) ; aucune vérification de convergence.

**L'usage.** Sur la machine de M26.6 sous $\pi_A$ (jamais d'entretien, $\gamma = 0{,}6$ ; tes $\mathbf{P}_{\pi_A}$ et $\mathbf{r}_{\pi_A}$ de M26.18), avec ta fonction :
- a) les lignes 1 à 3 depuis $\mathbf{V}_0 = \mathbf{0}$ (une liste de 3 listes de 3 nombres), à comparer à ta réponse a) de M26.7 ;
- b) le nombre $k$ d'itérations synchrones qu'il faut vraiment pour que $\lVert\mathbf{V}_k - \mathbf{V}^{\pi_A}\rVert_\infty \le 10^{-3}$ (le premier $k$ ; $\mathbf{V}^{\pi_A}$ par ta `bellman_linear_solve`), à comparer à ta réponse e) de M26.7 ; puis le premier $k$ où le critère a posteriori de M26.7 f) arrête l'itération ($\lVert\mathbf{V}_k - \mathbf{V}_{k-1}\rVert_\infty \le$ ton seuil) ; et vrai ou faux, l'erreur y est bien au plus $10^{-3}$ (une liste de 3 : les deux entiers, puis le booléen) ;
- c) écris `in_place_sweeps_19(P, r, gamma, V0, n_sweeps)`, qui rend le tableau de forme $(n_{\text{sweeps}} + 1, m)$ des balayages en place (les états dans l'ordre $0, 1, \ldots$, chaque nouvelle valeur servant aussitôt ; la ligne 0 est `V0`) : sa ligne 3 depuis $\mathbf{0}$, à comparer à ta réponse c) de M26.7, et le nombre de balayages qu'il faut pour $10^{-3}$ (une liste de 4 : les 3 nombres, puis l'entier).

La vérification essaie aussi `in_place_sweeps_19` sur deux chaînes que tu ne vois pas. Ce qu'il faut observer : la borne a priori est presque atteinte (l'écart garde une composante le long de $\mathbf{1}$, que $\gamma\mathbf{P}$ multiplie exactement par $\gamma$), et l'itération en place gagne quelques balayages.

In [ ]:
EPS_19 = 1e-3                                              # the accuracy of M26.7 e) and f)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M26")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M26, "test_bellman_fixed_point_iterates_")

**L'usage** : les itérés de la machine, le nombre d'itérations, et les balayages en place.

In [ ]:
def in_place_sweeps_19(P, r, gamma, V0, n_sweeps):
    """The (n_sweeps + 1, m) array of the in-place sweeps: in each sweep, for i = 0, 1, ..., m - 1 in this order,
    V[i] = r[i] + gamma * (P[i] @ V), each new value used at once; row 0 is V0."""
    raise NotImplementedError("in_place_sweeps_19 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M26_19a = answer_M26_19b = answer_M26_19c = ...
with wbm.attempt("M26.19"):          # your functions: ⏳ while they are not written
    answer_M26_19a = ...  # a) a list of 3 lists of 3 numbers, as computed: rows 1 to 3 from V0 = 0 (the machine under π_A)
    answer_M26_19b = ...  # b) a list of 3: the first k with ‖V_k − V‖∞ ≤ 1e-3; the first k where the a posteriori criterion stops; True or False: the error there is at most 1e-3
    answer_M26_19c = ...  # c) a list of 4: row 3 of in_place_sweeps_19 from 0 (3 numbers, as computed), then the number of sweeps for 1e-3

In [ ]:
for letter, answer in zip("abc", [answer_M26_19a, answer_M26_19b, answer_M26_19c]):
    check_answer(f"M26.19{letter}", answer)

def gauss_seidel_c19(P, r, gamma, V0, n):
    """The in-place sweeps in matrix form: (I − γ L) W_{k+1} = r + γ U W_k, L the strict lower part of P, U the rest."""
    P, r = np.asarray(P, dtype=float), np.asarray(r, dtype=float)
    lower, upper = np.tril(P, -1), np.triu(P)
    W = [np.asarray(V0, dtype=float)]
    for _ in range(n):
        W.append(linalg.solve_triangular(np.eye(len(r)) - gamma * lower, r + gamma * upper @ W[-1], lower=True))
    return np.array(W)


with checked("M26.19"):
    results_19 = []
    report_19 = reporter(results_19)
    rng_c19 = np.random.default_rng(6201)
    for m_c19, gamma_c19, ends_c19 in ((4, 0.85, False), (5, 0.7, True)):
        P_c19 = rng_c19.random((m_c19, m_c19)) * (rng_c19.random((m_c19, m_c19)) < 0.7) + np.eye(m_c19) * 0.1
        P_c19 = P_c19 / P_c19.sum(axis=1, keepdims=True) * (rng_c19.uniform(0.6, 0.95, (m_c19, 1)) if ends_c19 else 1)
        r_c19, V0_c19 = rng_c19.normal(0, 3, m_c19), rng_c19.normal(0, 5, m_c19)
        want_c19 = gauss_seidel_c19(P_c19, r_c19, gamma_c19, V0_c19, 6)
        got_c19 = in_place_sweeps_19(P_c19.copy(), r_c19.copy(), gamma_c19, V0_c19.copy(), 6)
        if matches(got_c19, want_c19, rtol=1e-10, atol=1e-10):
            continue
        sync_c19 = ref_markov.bellman_fixed_point_iterates(P_c19, r_c19, gamma_c19, V0_c19, 6)
        reverse_c19 = gauss_seidel_c19(P_c19[::-1, ::-1], r_c19[::-1], gamma_c19, V0_c19[::-1], 6)[:, ::-1]
        if matches(got_c19, sync_c19, rtol=1e-10, atol=1e-10):
            why_c19 = "ce sont les itérés synchrones : en place, chaque nouvelle valeur sert aussitôt aux états suivants"
        elif matches(got_c19, reverse_c19, rtol=1e-10, atol=1e-10):
            why_c19 = "tu balaies les états du dernier au premier : l'ordre est 0, 1, …, m − 1"
        elif matches(got_c19, want_c19[1:], rtol=1e-10, atol=1e-10):
            why_c19 = "il manque la ligne 0, V0 : le tableau a n_sweeps + 1 lignes"
        else:
            why_c19 = f"attendu un tableau de forme {want_c19.shape} de dernière ligne {shown(want_c19[-1])}, reçu {shown(got_c19)[:160]}"
        report_19(False, "", f"in_place_sweeps_19, une chaîne que tu ne vois pas ({m_c19} états, γ = {fr(gamma_c19, 2)}, "
                  f"V0 non nul, 6 balayages) : {why_c19}")
    report_19(True, "in_place_sweeps_19 : deux chaînes que tu ne vois pas (une stochastique, une qui peut finir), "
              "depuis un V0 non nul", "")
    got0_c19 = in_place_sweeps_19(np.array([[0.2, 0.8], [0.5, 0.5]]), np.array([1.0, -1.0]), 0.6, np.array([3.0, 4.0]), 0)
    report_19(matches(got0_c19, [[3.0, 4.0]]), "in_place_sweeps_19 : 0 balayage rend la seule ligne V0",
              f"in_place_sweeps_19 avec n_sweeps = 0 : attendu [[3.0, 4.0]] (la ligne V0), reçu {shown(got0_c19)}")

### Ex M26.20 — La loi stationnaire de trois façons, `np.linalg.eig` sur Pᵀ, `scipy.linalg.null_space` et la méthode de la puissance, sur le surfeur aléatoire d'un web de six pages 📦 ★ ⏱️ 20 min
**Objectif :** calculer une loi stationnaire par trois méthodes de bibliothèque et par la méthode de la puissance, et relier la vitesse de celle-ci à la deuxième valeur propre.  
**Prérequis :** fiche §2 ; M12c ; Ex M26.17 · **Parcours :** C

Un petit web a six pages, numérotées 1 à 6 (états 0 à 5). Les liens (`LINKS_20`) : $1 \to 2$, $1 \to 3$ ; $2 \to 4$ ; $3 \to 1$, $3 \to 4$, $3 \to 5$ ; $4 \to 1$, $4 \to 3$ ; $5 \to 4$, $5 \to 6$ ; la page 6 n'a aucun lien. Construis $\mathbf{L}$ (`L_20` : le surfeur choisit au hasard un lien de sa page ; sur la page 6, il va sur une page quelconque, uniformément) puis $\mathbf{P}_d = d\,\mathbf{L} + \frac{1 - d}{6}\mathbf{1}\mathbf{1}^\top$ avec $d = 0{,}85$ (`P_D_20`). Calcule sa loi stationnaire de trois façons :
1. par `np.linalg.eig(P_D_20.T)`, en prenant le vecteur propre de la valeur propre la plus proche de 1, sa partie réelle, divisée par sa somme ;
2. par `linalg.null_space(P_D_20.T - np.eye(6))` (`scipy.linalg`), qui doit rendre une seule colonne, divisée par sa somme ;
3. par la méthode de la puissance, dans `power_stationary_20(P, tol)`, qui part de la loi uniforme, fait $\mathbf{x} \leftarrow \mathbf{x}\mathbf{P}$ jusqu'à ce que $\lVert\mathbf{x}_{k+1} - \mathbf{x}_k\rVert_1 <$ `tol`, et rend le couple `(law, steps)` : la dernière loi calculée et le nombre de produits faits.

Relève :
- a) le PageRank des six pages, par ta `stationary_distribution` (une liste de 6 nombres ; 4 décimales dans ta copie) ;
- b) vrai ou faux : les trois calculs et ta `stationary_distribution` coïncident à $10^{-10}$ près ;
- c) le nombre de pas de `power_stationary_20(P_D_20, 1e-12)`, puis le plus petit nombre de pas $k$ pour lequel la borne $\lVert\mathbf{x}_k - \boldsymbol{\pi}\rVert_1 \le 2\,d^k$ de la fiche garantit $10^{-12}$ (une liste de 2 entiers) ;
- d) $\lvert\lambda_2(\mathbf{P}_d)\rvert$ et $d\,\lvert\lambda_2(\mathbf{L})\rvert$, par `np.linalg.eigvals` ($\lambda_2$ : la deuxième plus grande valeur propre en module) (une liste de 2 nombres ; 4 décimales dans ta copie) ;
- e) les numéros des pages (de 1 à 6), du plus grand PageRank au plus petit (une liste de 6 entiers).

La vérification compare aussi ta matrice `P_D_20` à celle du surfeur, et essaie `power_stationary_20` sur une chaîne que tu ne vois pas. Ce qu'il faut observer : trois méthodes, une seule loi ; la méthode de la puissance converge au rythme de $\lvert\lambda_2(\mathbf{P}_d)\rvert$, plus vite que ne le garantit $d$.

In [ ]:
LINKS_20 = {1: [2, 3], 2: [4], 3: [1, 4, 5], 4: [1, 3], 5: [4, 6], 6: []}   # page -> the pages it links to
D_20 = 0.85                                                # the damping factor

In [ ]:
def power_stationary_20(P, tol):
    """The power method: x = the uniform law, then x <- x @ P until ‖x_{k+1} − x_k‖₁ < tol; returns (law, steps), the last
    law computed and the number of products made."""
    raise NotImplementedError("power_stationary_20 n'est pas encore écrite (04_indices.md si tu bloques)")


L_20 = P_D_20 = answer_M26_20a = answer_M26_20b = answer_M26_20c = answer_M26_20d = answer_M26_20e = ...
with wbm.attempt("M26.20"):          # your stationary_distribution and power_stationary_20: ⏳ while they are not written
    L_20 = ...             # the (6, 6) matrix L of the random surfer (page 6: uniform)
    P_D_20 = ...           # P_d = d L + (1 − d)/6 1 1ᵀ, d = D_20
    answer_M26_20a = ...  # a) a list of 6 numbers, as computed: the PageRank of the six pages (your stationary_distribution)
    answer_M26_20b = ...  # b) True or False: eig, null_space, the power method and your stationary_distribution agree to 1e-10
    answer_M26_20c = ...  # c) a list of 2 integers: the steps of power_stationary_20(P_D_20, 1e-12), then the smallest k with 2 d^k ≤ 1e-12
    answer_M26_20d = ...  # d) a list of 2 numbers, as computed: |λ2(P_d)| and d |λ2(L)|
    answer_M26_20e = ...  # e) a list of 6 integers: the pages (1 to 6), from the largest PageRank to the smallest

In [ ]:
for letter, answer in zip("abcde", [answer_M26_20a, answer_M26_20b, answer_M26_20c, answer_M26_20d, answer_M26_20e]):
    check_answer(f"M26.20{letter}", answer)

with checked("M26.20"):
    L_c20 = np.zeros((6, 6))                                  # the reference matrix of the surfer
    for page_c20, targets_c20 in LINKS_20.items():
        L_c20[page_c20 - 1] = np.full(6, 1 / 6) if not targets_c20 else np.isin(np.arange(1, 7), targets_c20) / len(targets_c20)
    if filled(P_D_20):
        same("M26.20", "P_D_20, la matrice du surfeur avec sauts", P_D_20, D_20 * L_c20 + (1 - D_20) / 6)
    else:
        print("⏳ Ex M26.20 : la comparaison attend ta matrice P_D_20.")
    rng_c20 = np.random.default_rng(6202)
    P_c20 = rng_c20.random((5, 5)) + 0.05
    P_c20 = P_c20 / P_c20.sum(axis=1, keepdims=True)
    x_c20, steps_c20 = np.full(5, 0.2), 0                     # the power method of the statement, step by step
    while True:
        y_c20, steps_c20 = x_c20 @ P_c20, steps_c20 + 1
        if np.abs(y_c20 - x_c20).sum() < 1e-10:
            break
        x_c20 = y_c20
    law_c20, n_c20 = pair("M26.20", "power_stationary_20", power_stationary_20(P_c20.copy(), 1e-10), names=("law", "steps"))
    if not same("M26.20", "power_stationary_20, une chaîne que tu ne vois pas (tol = 1e-10) : la loi", law_c20,
                y_c20, rtol=0, atol=1e-9):
        raise StopCheck
    if isinstance(n_c20, (bool, np.bool_)) or not isinstance(n_c20, (int, np.integer)):
        verdict("M26.20", False, "", f"power_stationary_20 doit rendre le nombre de pas comme un entier, reçu {n_c20!r}")
    elif n_c20 in (steps_c20 - 1, steps_c20 + 1):
        verdict("M26.20", False, "", f"power_stationary_20 compte {int(n_c20)} pas au lieu de {steps_c20} : compte chaque "
                "produit x ← xP, le dernier compris (celui après lequel l'écart passe sous tol)")
    else:
        verdict("M26.20", int(n_c20) == steps_c20, f"power_stationary_20 : {steps_c20} pas sur une chaîne que tu ne vois pas, "
                "comme attendu", f"power_stationary_20 : attendu {steps_c20} pas, reçu {int(n_c20)}")

**Après la vérification**, dans ta copie : pourquoi la page 6, sans lien, reçoit-elle si peu ? Combien de pas la méthode de la puissance gagne-t-elle sur la borne $2\,d^k$, et quel nombre fixe vraiment sa vitesse ?

## Partie B · Prédire, mesurer, dessiner

Fiche §2, §5 et §8. La loi d'occupation actualisée, prédite puis calculée ; le biais du maximum, simulé ; l'oubli du départ, dessiné.

### Ex M26.21 — Prédire la somme des lignes de (1 − γ)(I − γP)⁻¹, puis leur limite quand γ tend vers 1, et vérifier 🔮 ★ ⏱️ 15 min
**Objectif :** prévoir, puis vérifier, que la loi d'occupation actualisée est une loi pour tout γ, qu'elle oublie le départ quand γ tend vers 1, et qu'elle reste sur le départ quand γ tend vers 0.  
**Prérequis :** fiche §2 et §5 ; Ex M26.2, M26.17 et M26.18 · **Parcours :** C, E

On reprend les trois opérateurs de M26.2 (`P_OPERATORS`) et $\mathbf{M}_\gamma = (1 - \gamma)(\mathbf{I} - \gamma\mathbf{P})^{-1}$, dont la ligne $s$ est la loi d'occupation actualisée depuis $s$ (fiche §5). **Avant d'exécuter quoi que ce soit**, prédis :
- a) la somme de chaque ligne de $\mathbf{M}_\gamma$ : (A) 1 ; (B) $\frac{1}{1 - \gamma}$ ; (C) elle dépend de la ligne (une lettre) ;
- b) quand $\gamma$ tend vers 1, chaque ligne de $\mathbf{M}_\gamma$ tend vers : (A) la loi concentrée sur l'état de départ ; (B) la loi stationnaire $\boldsymbol{\pi}$ ; (C) la loi uniforme ; (D) le vecteur nul (une lettre) ;
- c) quand $\gamma$ tend vers 0, $\mathbf{M}_\gamma$ tend vers : (A) $\mathbf{I}$ ; (B) $\mathbf{P}$ ; (C) la matrice dont chaque ligne est $\boldsymbol{\pi}$ (une lettre).

Puis, une fois tes prédictions écrites et vérifiées, calcule $\mathbf{M}_\gamma$ colonne par colonne avec ta `bellman_linear_solve` (la colonne $j$ est $(1 - \gamma)$ fois la solution pour $\mathbf{r} = \mathbf{e}_j$), pour $\gamma = 0{,}6$, $0{,}7$, $0{,}85$ et $0{,}96$ (`GAMMAS_21` ; `matrices_21`, la liste des quatre matrices), et $\boldsymbol{\pi}$ avec ta `stationary_distribution`. Relève :
- d) le plus grand écart à 1 des sommes des lignes, sur les quatre valeurs de $\gamma$ (un nombre, tel que calculé) ;
- e) pour chaque $\gamma$, le plus grand écart $\max_{s, s'}\lvert M_\gamma(s, s') - \pi_{s'}\rvert$ (une liste de 4 nombres ; 4 décimales dans ta copie) ;
- f) vrai ou faux : la limite exacte de $\mathbf{M}_\gamma$ quand $\gamma \to 1$, calculée par SymPy (la matrice en fractions `P_SYMBOLIC_21`, un symbole `gamma`, `sp.limit` sur chaque coefficient), a toutes ses lignes égales à $\boldsymbol{\pi}$ ;
- g) la limite exacte quand $\gamma \to 0$, par le même calcul (une matrice $3 \times 3$).

La vérification de l'expérience ne s'exécute qu'une fois tes trois prédictions écrites : elle compare tes quatre matrices à $(1 - \gamma)(\mathbf{I} - \gamma\mathbf{P})^{-1}$. Ce qu'il faut observer : les lignes sont des lois pour tout $\gamma$ ; quand l'arrêt est rare, la loi de l'état où il tombe oublie le départ et devient $\boldsymbol{\pi}$ ; quand il est immédiat, elle reste sur le départ.

In [ ]:
GAMMAS_21 = [0.6, 0.7, 0.85, 0.96]                        # the four discount factors
P_SYMBOLIC_21 = sp.Matrix([[sp.Rational(9, 10), sp.Rational(1, 20), sp.Rational(1, 20)],   # P_OPERATORS in fractions
                           [sp.Rational(1, 20), sp.Rational(17, 20), sp.Rational(1, 10)],
                           [sp.Rational(1, 5), sp.Rational(3, 20), sp.Rational(13, 20)]])

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M26_21a = ...  # a) "A", "B" or "C", written BEFORE the experiment: the sum of each row of M_γ...
prediction_M26_21b = ...  # b) "A", "B", "C" or "D": when γ tends to 1, each row of M_γ tends to...
prediction_M26_21c = ...  # c) "A", "B" or "C": when γ tends to 0, M_γ tends to...

In [ ]:
for letter, prediction in zip("abc", [prediction_M26_21a, prediction_M26_21b, prediction_M26_21c]):
    wbm.check(f"M26.21{letter}", prediction)

**L'expérience.** Écris-la une fois tes trois prédictions vérifiées ; sa vérification ne s'exécute qu'à cette condition.

In [ ]:
matrices_21 = answer_M26_21d = answer_M26_21e = answer_M26_21f = answer_M26_21g = ...
with wbm.attempt("M26.21"):          # your bellman_linear_solve and stationary_distribution: ⏳ while they are not written
    matrices_21 = ...      # the four (3, 3) matrices M_γ, for γ in GAMMAS_21, column by column with your bellman_linear_solve
    answer_M26_21d = ...  # d) a number, as computed: the largest |sum of a row − 1|, over the rows and the four γ
    answer_M26_21e = ...  # e) a list of 4 numbers, as computed: for each γ, max |M_γ(s, s′) − π_s′|
    answer_M26_21f = ...  # f) True or False: the exact limit when γ → 1 (SymPy) has every row equal to π
    answer_M26_21g = ...  # g) a 3 × 3 matrix: the exact limit when γ → 0 (SymPy)

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M26_21a, prediction_M26_21b, prediction_M26_21c]):
    print("⏳ Écris d'abord tes trois prédictions, puis exécute à nouveau.")
else:
    with checked("M26.21"):
        if not filled(matrices_21):
            print("⏳ Ex M26.21 : la vérification attend tes quatre matrices matrices_21.")
            raise StopCheck
        if not isinstance(matrices_21, (list, tuple)) or len(matrices_21) != 4:
            verdict("M26.21", False, "", "matrices_21 doit être une liste de 4 matrices (3, 3), une par γ de GAMMAS_21")
            raise StopCheck
        for gamma_c21, got_c21 in zip(GAMMAS_21, matrices_21):
            want_c21 = (1 - gamma_c21) * np.linalg.inv(np.eye(3) - gamma_c21 * P_OPERATORS)
            if matches(got_c21, want_c21 / (1 - gamma_c21)):
                verdict("M26.21", False, "", f"γ = {fr(gamma_c21, 2)} : la colonne j de M_γ est (1 − γ) fois la solution pour "
                        "r = e_j : il manque le facteur (1 − γ)")
                raise StopCheck
            if matches(got_c21, want_c21.T):
                verdict("M26.21", False, "", f"γ = {fr(gamma_c21, 2)} : ta matrice est transposée : la solution pour r = e_j "
                        "est la colonne j de M_γ")
                raise StopCheck
            if not same("M26.21", f"M_γ pour γ = {fr(gamma_c21, 2)}", got_c21, want_c21, rtol=1e-9, atol=1e-12):
                raise StopCheck
        print("   sommes des lignes :", [np.round(np.asarray(M, dtype=float).sum(axis=1), 12).tolist() for M in matrices_21][-1],
              "(γ = 0,96)")

    for letter, answer in zip("efg", [answer_M26_21e, answer_M26_21f, answer_M26_21g]):
        check_answer(f"M26.21{letter}", answer)

    with checked("M26.21"):
        if not filled(answer_M26_21d):
            print("⏳ Ex M26.21d : pas encore fait.")
        else:
            try:
                value_c21 = float(answer_M26_21d)
            except (TypeError, ValueError):
                value_c21 = float("nan")
            if 0 <= value_c21 < 1e-12:
                print(f"✅ Ex M26.21d : {value_c21:.1e}, une erreur d'arrondi : les lignes ont pour somme 1.")
            elif abs(value_c21 - (1 / (1 - 0.96) - 1)) < 1e-6:
                print("❌ Ex M26.21d : tes sommes de lignes valent 1/(1 − γ) : la colonne j de M_γ est (1 − γ) fois la solution "
                      "pour r = e_j.")
            else:
                print(f"❌ Ex M26.21d : attendu un écart de l'ordre de l'arrondi (moins de 1e-12, positif ou nul), reçu "
                      f"{answer_M26_21d!r} : prends le plus grand |somme d'une ligne − 1|, sur les lignes et les quatre γ.")

**Après l'expérience**, dans ta copie : pourquoi les lignes de $\mathbf{M}_\gamma$ ont-elles pour somme 1 ? Pourquoi l'écart à $\boldsymbol{\pi}$ ne baisse-t-il que lentement quand $\gamma$ approche 1 ? Que devient la loi de l'état à l'arrêt quand l'arrêt est immédiat ?

### Ex M26.22 — Le biais du maximum par simulation : k fournisseurs de même qualité, n mesures chacun, le meilleur mesuré contre l'estimateur double 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer par simulation le biais du maximum de k moyennes bruitées, sa croissance avec k et sa décroissance en 1/√n, et voir l'estimateur double rester sans biais, puis devenir pessimiste.  
**Prérequis :** fiche §8 ; M02c ; Ex M26.9 · **Parcours :** C

$k$ fournisseurs ont la même durée de vie moyenne, 1 000 h (`MU_22`), et l'on teste $n$ batteries de chacun (écart-type 30 h par batterie, `SIGMA0_22`). Écris `max_bias_22(k, n, delta, n_rep, rng)`, qui simule `n_rep` expériences, toutes en une fois avec des tableaux : les moyennes vraies valent 1 000 h, sauf celle du fournisseur 0, qui vaut $1\,000 + \delta$ ; dans chaque expérience, on tire les $n \times k$ durées (`rng.normal`, un tableau de forme `(n_rep, n, k)`), l'estimateur naïf est la plus grande des $k$ moyennes mesurées, et l'estimateur double remesure le fournisseur choisi sur $n$ nouvelles batteries (un tableau de forme `(n_rep, n)`, tiré après le premier) ; la fonction rend le couple `(naive, double)` des deux biais moyens, par rapport à la meilleure moyenne vraie $1\,000 + \max(\delta, 0)$. Écris aussi `draw_bias_22(ax, ks, ns, table)`, qui trace dans l'axe reçu le biais naïf contre $k$, une courbe par $n$ (la colonne $j$ de `table` pour `ns[j]`), avec des noms d'axes et une légende.

Avec `rng = np.random.default_rng(6222)`, `n_rep = 20000` (`N_REP_22`), et ce même `rng` pour tous les appels, dans l'ordre : $k$ parcourt $(2, 3, 5, 10)$ et, pour chaque $k$, $n$ parcourt $(4, 16, 64)$ (`KS_22`, `NS_22`), avec $\delta = 0$ ; puis un dernier appel avec $k = 3$, $n = 16$ et $\delta = 5$. Relève :
- a) le tableau des biais naïfs, une ligne par $k$, une colonne par $n$ (une liste de 4 listes de 3 nombres ; 2 décimales dans ta copie) ;
- b) vrai ou faux : pour $k = 2$ et $n = 16$, le biais naïf est à moins de 0,2 h de $\frac{\sigma}{\sqrt\pi}$, avec $\sigma = 30/\sqrt{16}$ ;
- c) pour $k = 3$ et $n = 16$, le biais naïf divisé par $\sigma$, à comparer à $\frac{3}{2\sqrt\pi} \approx 0{,}846$ (l'espérance du maximum de trois variables indépendantes de loi $\mathcal{N}(0, 1)$, admise) (un nombre ; 3 décimales dans ta copie) ;
- d) vrai ou faux : dans tout le tableau, le biais de l'estimateur double est inférieur à 0,3 h en valeur absolue ;
- e) avec $k = 3$, $n = 16$ et $\delta = 5$ : les deux biais, naïf puis double (une liste de 2 nombres ; 2 décimales dans ta copie).

La vérification essaie `max_bias_22` sur un cas que tu ne vois pas, puis appelle `draw_bias_22` sur le tableau attendu et affiche la figure. Ce qu'il faut conclure : le biais grandit avec $k$ et décroît comme $1/\sqrt n$ (il est divisé par 4 quand $n$ est multiplié par 16) ; l'estimateur double est sans biais quand tout se vaut, et pessimiste quand une option est meilleure.

In [ ]:
SIGMA0_22 = 30.0                                           # the standard deviation of one battery (h)
MU_22 = 1000.0                                             # the true mean life of every supplier (h)
KS_22, NS_22 = (2, 3, 5, 10), (4, 16, 64)                  # the numbers of suppliers and of batteries per supplier
N_REP_22 = 20000                                           # the number of simulated experiments

In [ ]:
def max_bias_22(k, n, delta, n_rep, rng):
    """n_rep experiments at once: the true means are MU_22, except that of supplier 0, MU_22 + delta; the n × k lives by
    rng.normal of shape (n_rep, n, k); the naive estimate is the largest of the k measured means; the double estimate
    measures the chosen supplier again on n new batteries (shape (n_rep, n), drawn after the first). Returns the pair
    (naive, double) of the mean biases, measured from the best true mean MU_22 + max(delta, 0)."""
    raise NotImplementedError("max_bias_22 n'est pas encore écrite (04_indices.md si tu bloques)")


def draw_bias_22(ax, ks, ns, table):
    """In ax: the naive bias against k, one curve per n (column j of table for ns[j]); axis labels and a legend. No
    other axes."""
    raise NotImplementedError("draw_bias_22 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M26_22a = answer_M26_22b = answer_M26_22c = answer_M26_22d = answer_M26_22e = ...
with wbm.attempt("M26.22"):          # your max_bias_22: ⏳ while it is not written
    answer_M26_22a = ...  # a) a list of 4 lists of 3 numbers, as computed: the naive biases, one row per k, one column per n
    answer_M26_22b = ...  # b) True or False: for k = 2 and n = 16, the naive bias is within 0.2 h of σ/√π, σ = 30/√16
    answer_M26_22c = ...  # c) a number, as computed: the naive bias for k = 3 and n = 16, divided by σ
    answer_M26_22d = ...  # d) True or False: in the whole table, |the bias of the double estimate| < 0.3 h
    answer_M26_22e = ...  # e) a list of 2 numbers, as computed: the naive bias, then the double one, for k = 3, n = 16, δ = 5

In [ ]:
def max_bias_c22(k, n, delta, n_rep, rng):
    """The procedure of the statement, by the check (never your code)."""
    means = np.full(k, MU_22)
    means[0] += delta
    first = rng.normal(means, SIGMA0_22, size=(n_rep, n, k)).mean(axis=1)
    chosen = np.argmax(first, axis=1)
    second = rng.normal(means[chosen][:, None], SIGMA0_22, size=(n_rep, n)).mean(axis=1)
    best = MU_22 + max(delta, 0.0)
    return float(first.max(axis=1).mean() - best), float(second.mean() - best)


def table_hint_22(value):
    """a): a table close to the right one, not equal: the order of the draws."""
    want = [[max_bias_c22(k, n, 0.0, 2000, np.random.default_rng(6223))[0] for n in (4, 16)] for k in (2, 3)]
    got = np.asarray(value, dtype=float)
    if got.shape == (4, 3) and np.all(np.abs(got[:2, :2] - np.asarray(want)) < 2.0):
        return ("tes biais ont le bon ordre de grandeur : les valeurs exactes demandent les tirages dans l'ordre de "
                "l'énoncé (k, puis n, puis le dernier appel), avec le même rng, rng.normal de forme (n_rep, n, k) puis "
                "(n_rep, n)")
    return None


check_answer("M26.22a", answer_M26_22a, hint=table_hint_22)
check_answer("M26.22b", answer_M26_22b)
check_answer("M26.22c", answer_M26_22c)
check_answer("M26.22d", answer_M26_22d)
check_answer("M26.22e", answer_M26_22e)

with checked("M26.22"):
    results_22 = []
    report_22 = reporter(results_22)
    want_c22 = max_bias_c22(3, 8, 4.0, 3000, np.random.default_rng(6223))
    got_c22 = pair("M26.22", "max_bias_22", max_bias_22(3, 8, 4.0, 3000, np.random.default_rng(6223)),
                   names=("naive", "double"))
    if not matches(got_c22, want_c22, rtol=1e-9, atol=1e-9):
        shifted_c22 = (want_c22[0] + 4.0, want_c22[1] + 4.0)
        report_22(False, "", "max_bias_22(3, 8, 4.0, 3000, default_rng(6223)) : attendu " + shown(want_c22) + ", reçu "
                  + shown(got_c22) + (" : les biais se mesurent par rapport à la meilleure moyenne vraie, 1 000 + max(δ, 0)"
                                      if matches(got_c22, shifted_c22, rtol=1e-9, atol=1e-9) else
                                      " : tire les mesures dans l'ordre de l'énoncé, (n_rep, n, k) puis (n_rep, n)"))
    report_22(True, "max_bias_22 : un cas que tu ne vois pas (k = 3, n = 8, δ = 4, 3 000 expériences), comme attendu", "")
    rng_c22 = np.random.default_rng(6222)                     # one rng for the whole table, as in the statement
    table_c22 = [[max_bias_c22(k, n, 0.0, N_REP_22, rng_c22)[0] for n in NS_22] for k in KS_22]
    fig_22, ax_22 = plt.subplots(figsize=(6.5, 4))
    try:
        opened_22 = drawn(draw_bias_22, ax_22, list(KS_22), list(NS_22), np.array(table_c22))
        report_22(not opened_22, "draw_bias_22 dessine dans l'axe reçu", "draw_bias_22 doit dessiner dans l'axe reçu, "
                  "sans ouvrir de figure")
        missing_22 = [f"n = {n}" for j, n in enumerate(NS_22)
                      if not through(ax_22, KS_22, np.array(table_c22)[:, j], rtol=1e-9)]
        report_22(not missing_22, "une courbe du biais naïf contre k pour chaque n",
                  "il manque la courbe du biais naïf contre k pour " + ", ".join(missing_22))
        report_22(ax_22.get_legend() is not None and ax_22.get_xlabel() != "" and ax_22.get_ylabel() != "",
                  "des noms d'axes et une légende", "draw_bias_22 doit nommer les axes et mettre une légende (un label "
                  "par courbe)")
    except StopCheck:
        fig_22.tight_layout()
        plt.show()
        raise
    except BaseException:
        plt.close(fig_22)
        raise
    fig_22.tight_layout()
    plt.show()
    verdict("M26.22", all(results_22), "ta simulation et ta figure sont justes.", "")

**Après la figure**, dans ta copie : pourquoi le biais grandit-il avec $k$ ? Pourquoi est-il divisé par deux quand $n$ est multiplié par quatre ? Pourquoi l'estimateur double devient-il pessimiste quand un fournisseur est meilleur ?

### Ex M26.23 — La loi d'une chaîne au fil des pas depuis trois départs, et ‖μ_n − π‖₁ en échelle logarithmique contre |λ₂|ⁿ 🎨 ★ ⏱️ 15 min
**Objectif :** voir la loi d'une chaîne oublier son départ, et lire sur une échelle logarithmique que l'écart décroît comme |λ₂|ⁿ.  
**Prérequis :** fiche §1 et §2 ; M12c ; Ex M26.2, M26.16 et M26.17 · **Parcours :** C

Écris `draw_convergence_23(ax_law, ax_dist, P, starts, n_max)`, qui dessine sur les **axes reçus** (sans ouvrir de figure), avec ta `n_step_distribution` et ta `stationary_distribution` :
- dans `ax_law`, la probabilité d'être dans l'état 0 en fonction de $n$, de 0 à `n_max`, une courbe par état de départ de `starts`, et la droite horizontale $\pi_0$ ;
- dans `ax_dist`, en échelle logarithmique, $\lVert\boldsymbol{\mu}_n - \boldsymbol{\pi}\rVert_1$ pour chaque départ ($n = 0, \ldots,$ `n_max`), et en pointillés la courbe $\lvert\lambda_2\rvert^n$ ($\lambda_2$ calculée par `np.linalg.eigvals` : la deuxième plus grande valeur propre en module) ;
- des titres, des noms d'axes et une légende dans chaque panneau.

La vérification l'appelle sur les trois opérateurs de M26.2 (`P_OPERATORS`), avec `starts = [0, 1, 2]` et `n_max = 100` (`STARTS_23`, `N_MAX_23`), et affiche la figure ; elle l'appelle aussi sur une chaîne que tu ne vois pas, et regarde chaque dessin. Tes propres calculs servent à relever :
- a) $\lvert\lambda_2\rvert$ (un nombre ; 4 décimales dans ta copie) ;
- b) la pente de $n \mapsto \ln\lVert\boldsymbol{\mu}_n - \boldsymbol{\pi}\rVert_1$ entre $n = 50$ et $n = 100$, depuis l'état 0 (un nombre ; 4 décimales dans ta copie), à comparer à $\ln\lvert\lambda_2\rvert$ ;
- c) le premier $n$ où les trois écarts sont tous inférieurs à $10^{-6}$ (un entier).

Ce qu'il faut observer : après quelques pas, les trois courbes deviennent des droites parallèles de pente $\ln\lvert\lambda_2\rvert$ ; le départ ne change que leur hauteur.

In [ ]:
STARTS_23 = [0, 1, 2]                                      # the three starting states (the three operators)
N_MAX_23 = 100

In [ ]:
def draw_convergence_23(ax_law, ax_dist, P, starts, n_max):
    """With your n_step_distribution and stationary_distribution. In ax_law: P(S_n = 0) against n = 0, ..., n_max, one
    curve per starting state of starts, and the horizontal line π_0; in ax_dist, in log scale: ‖μ_n − π‖₁ for each start,
    and |λ2|ⁿ dashed (np.linalg.eigvals); titles, axis labels and a legend in each panel. No other axes."""
    raise NotImplementedError("draw_convergence_23 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M26_23a = answer_M26_23b = answer_M26_23c = ...
with wbm.attempt("M26.23"):          # your library: ⏳ while it is not written
    answer_M26_23a = ...  # a) a number, as computed: |λ2| of P_OPERATORS
    answer_M26_23b = ...  # b) a number, as computed: the slope of n ↦ ln ‖μ_n − π‖₁ between n = 50 and n = 100, from state 0
    answer_M26_23c = ...  # c) an integer: the first n where the three gaps are all below 1e-6

In [ ]:
for letter, answer in zip("abc", [answer_M26_23a, answer_M26_23b, answer_M26_23c]):
    check_answer(f"M26.23{letter}", answer)

def look_23(ax_law, ax_dist, P, starts, n_max):
    """What is wrong in a drawing of draw_convergence_23 (expected values by the reference): French phrases."""
    P = np.asarray(P, dtype=float)
    pi = ref_markov.stationary_distribution(P)
    steps = np.arange(n_max + 1)
    lam2 = np.sort(np.abs(np.linalg.eigvals(P)))[::-1][1]
    missing = []
    for s in starts:
        laws = np.array([ref_markov.n_step_distribution(P, np.eye(len(P))[s], n) for n in steps])
        columns = np.array([np.linalg.matrix_power(P, n)[:, s] for n in steps])   # (wrong) P^n e_s, the column convention
        if not through(ax_law, steps, laws[:, 0]):
            missing.append(f"la probabilité d'être dans l'état 0 depuis l'état {s}, contre n = 0, …, {n_max}"
                           + (" (tu traces la convention colonne, Pⁿ e_s : une loi avance par mu @ P)"
                              if through(ax_law, steps, columns[:, 0]) else ""))
        dist = np.abs(laws - pi).sum(axis=1)
        if not through(ax_dist, steps, dist):
            wrong = np.abs(columns - pi).sum(axis=1)
            missing.append(f"‖μ_n − π‖₁ depuis l'état {s}" + (" (tes courbes ne tendent pas vers 0 : c'est la convention "
                           "colonne, Pⁿ e_s ; une loi avance par mu @ P)" if through(ax_dist, steps, wrong) else ""))
    if not horizontal_at(ax_law, pi[0]):
        missing.append("la droite horizontale π₀ dans le panneau de gauche")
    if not through(ax_dist, steps, lam2 ** steps, dashed=True):
        missing.append("la courbe |λ₂|ⁿ en pointillés dans le panneau de droite")
    if ax_dist.get_yscale() != "log":
        missing.append("l'échelle logarithmique du panneau de droite (ax.set_yscale(\"log\"))")
    for name, ax in (("de gauche", ax_law), ("de droite", ax_dist)):
        if not ax.get_title() or not ax.get_xlabel() or not ax.get_ylabel() or ax.get_legend() is None:
            missing.append(f"un titre, des noms d'axes et une légende dans le panneau {name}")
    return missing


with checked("M26.23"):
    results_23 = []
    report_23 = reporter(results_23)
    hidden_c23 = np.array([[0.5, 0.3, 0.2, 0.0], [0.1, 0.6, 0.1, 0.2], [0.25, 0.0, 0.55, 0.2], [0.3, 0.1, 0.0, 0.6]])
    fig_c23, (law_c23, dist_c23) = plt.subplots(1, 2)
    try:
        with own_code("M26.23", "draw_convergence_23", REF_MARKOV):
            opened_c23 = drawn(draw_convergence_23, law_c23, dist_c23, hidden_c23.copy(), [3, 0], 40)
        missing_c23 = look_23(law_c23, dist_c23, hidden_c23, [3, 0], 40)
    finally:
        plt.close(fig_c23)
    fig_23, (law_23, dist_23) = plt.subplots(1, 2, figsize=(12, 4.2))
    try:
        with own_code("M26.23", "draw_convergence_23", REF_MARKOV):
            opened_23 = drawn(draw_convergence_23, law_23, dist_23, P_OPERATORS.copy(), list(STARTS_23), N_MAX_23)
        report_23(not opened_23 and not opened_c23, "draw_convergence_23 dessine dans les axes reçus",
                  "draw_convergence_23 doit dessiner dans les deux axes reçus, sans ouvrir de figure")
        missing_23 = look_23(law_23, dist_23, P_OPERATORS, STARTS_23, N_MAX_23)
        report_23(not missing_23, "les trois opérateurs : les trois départs, π₀, les écarts en échelle logarithmique et "
                  "|λ₂|ⁿ sont là", "les trois opérateurs : il manque " + " ; ".join(missing_23))
        report_23(not missing_c23, "une chaîne que tu ne vois pas (4 états, deux départs, 40 pas) est dessinée juste aussi",
                  "une chaîne que tu ne vois pas (4 états, départs [3, 0], 40 pas) : il manque " + " ; ".join(missing_c23))
    except StopCheck:
        fig_23.tight_layout()
        plt.show()
        raise
    except BaseException:
        plt.close(fig_23)
        raise
    fig_23.tight_layout()
    plt.show()
    verdict("M26.23", all(results_23), "la loi au fil des pas et l'oubli du départ sont dessinés.", "")

**Après la figure**, dans ta copie : pourquoi les trois courbes de droite deviennent-elles parallèles ? Que changerait une deuxième valeur propre de module plus petit ? Combien de mois faut-il pour que la loi des parts de marché ne dépende presque plus d'aujourd'hui ?

## Partie C · Déboguer, tester

Fiche §1, §2, §5 et §6. Quatre erreurs classiques, qui passent une vérification mal choisie ; un test de propriété des lois stationnaires et des itérés de Bellman. Les données et les vérifications s'appuient sur la référence (`ref_markov`), pour ne pas dépendre de ton code.

### Ex M26.24 — Chasse à l'erreur : une matrice lue dans la mauvaise convention, une loi stationnaire non normalisée, un état final qui rapporte encore, γ = 1 sans fin 🐛 ★ ⏱️ 25 min
**Objectif :** reconnaître quatre erreurs classiques sur les chaînes de Markov et le système de Bellman, qui passent une vérification mal choisie, puis les corriger.  
**Prérequis :** fiche §1, §2 et §5 ; Ex M26.1, M26.2, M26.4 et M26.16 à M26.18 · **Parcours :** C, E

Un collègue a écrit quatre fonctions, chacune accompagnée d'une vérification qui passe (cellule fournie) :
1. `bikes_after(P, mu0, n)` doit rendre la loi d'un vélo après `n` jours ; elle répète `n` fois `mu = P @ mu`. Il l'a vérifiée sur la matrice $\mathbf{S}$ (`S_24`), depuis la station 0, sur 5 jours : le résultat a pour somme 1 et égale la ligne 0 de `np.linalg.matrix_power(S, 5)`. Il l'applique aux vélos de M26.1.
2. `market_shares(P)` doit rendre la loi stationnaire ; elle calcule `w, v = np.linalg.eig(P.T)`, prend la colonne de `v` dont la valeur propre est la plus proche de 1, et rend sa partie réelle. Il l'a vérifiée sur les trois opérateurs de M26.2 : `np.allclose(pi @ P, pi)` vaut `True`.
3. `subscription_value(P, monthly_fee, closing_fee, gamma)` doit rendre les valeurs d'un abonnement à trois états, essai (0), payant (1) et résilié (2, absorbant), où un abonné payant verse `monthly_fee` chaque mois et où une résiliation coûte une fois `closing_fee` à l'abonné ; elle construit $\mathbf{r} = (0,\ \text{monthly\_fee},\ \text{closing\_fee})$ et rend `np.linalg.solve(np.eye(3) - gamma * P, r)`. Il l'a vérifiée avec `P_SUBSCRIPTION_24`, 12 € par mois, 15 € de frais et $\gamma = 0{,}96$ : le résidu $\lVert\mathbf{V} - (\mathbf{r} + \gamma\mathbf{P}\mathbf{V})\rVert_\infty$ est inférieur à $10^{-9}$.
4. `total_takings(P, r)` doit rendre la recette totale moyenne jusqu'à la fin, depuis chaque état ; elle rend `np.linalg.solve(np.eye(m) - P, r)`. Il l'a vérifiée sur le cursus de M26.4 (`P` = les trois années, `r` = des 1) : elle redonne les durées moyennes. Il l'applique ensuite aux vélos, avec une matrice estimée sur 100 trajets par station (`P_ESTIMATED_24`, de lignes $(61, 29, 10)/100$, $(33, 41, 26)/100$ et $(38, 23, 39)/100$), et les recettes journalières $(1{,}5 ;\ 1 ;\ 0{,}5)$ € (`TAKINGS_24`).

À faire : trouve et corrige chaque erreur ; les fonctions corrigées s'appellent `bikes_after_fixed`, `market_shares_fixed`, `subscription_value_fixed` et `total_takings_fixed`, avec les mêmes arguments (`total_takings_fixed` lève une `ValueError` quand le processus ne finit pas avec certitude, testé par `np.linalg.matrix_rank`). Puis réponds :
- a) la loi des vélos de M26.1 après trois jours par `bikes_after_fixed`, et la somme des coordonnées de ce que rend `bikes_after` (une liste de 4 nombres ; 4 décimales dans ta copie) ;
- b) la loi rendue par `market_shares_fixed` pour les trois opérateurs, et la valeur absolue de la somme de ce que rend `market_shares` (une liste de 4 nombres ; 4 décimales dans ta copie) ;
- c) les valeurs que rend `subscription_value_fixed` pour le modèle du collègue, et celle que `subscription_value` donne à l'état résilié (une liste de 4 nombres ; 2 décimales dans ta copie) ;
- d) vrai ou faux : `total_takings_fixed` lève une `ValueError` sur la matrice des vélos (`P_ESTIMATED_24`).

Dans ta copie : ce que rend `total_takings` sur cette matrice (une erreur, ou des nombres : lesquels ?), et, pour chacune des quatre erreurs, pourquoi la vérification du collègue ne la voyait pas. La vérification essaie tes corrections sur des cas que tu ne vois pas : des chaînes non symétriques, d'autres lois stationnaires, d'autres abonnements, une fin sûre et deux fins qui ne le sont pas.

In [ ]:
S_24 = np.array([[0.7, 0.2, 0.1], [0.2, 0.6, 0.2], [0.1, 0.2, 0.7]])      # the colleague's test matrix
P_SUBSCRIPTION_24 = np.array([[0, 0.55, 0.45], [0, 0.9, 0.1], [0, 0, 1.0]])   # trial (0), paying (1), cancelled (2)
COUNTS_BIKES_24 = np.array([[61, 29, 10], [33, 41, 26], [38, 23, 39]])        # 100 trips from each station
P_ESTIMATED_24 = COUNTS_BIKES_24 / 100
TAKINGS_24 = np.array([1.5, 1.0, 0.5])                    # the daily takings of a bike in A, B, C (€)


# The colleague's code
def bikes_after(P, mu0, n):
    """The law of a bike after n days."""
    mu = np.array(mu0, dtype=float)
    for _ in range(n):
        mu = P @ mu
    return mu


def market_shares(P):
    """The stationary law."""
    w, v = np.linalg.eig(np.asarray(P, dtype=float).T)
    return np.real(v[:, np.argmin(np.abs(w - 1))])


def subscription_value(P, monthly_fee, closing_fee, gamma):
    """The values of a subscription: trial (0), paying (1) who pays monthly_fee each month, cancelled (2, absorbing);
    a cancellation costs closing_fee once to the subscriber."""
    r = np.array([0.0, monthly_fee, closing_fee])
    return np.linalg.solve(np.eye(3) - gamma * np.asarray(P, dtype=float), r)


def total_takings(P, r):
    """The mean total takings until the end, from each state."""
    m = len(r)
    return np.linalg.solve(np.eye(m) - np.asarray(P, dtype=float), np.asarray(r, dtype=float))


# The colleague's checks: all pass
print("(1) S symétrique, 5 jours : somme 1 :", bool(np.isclose(bikes_after(S_24, [1, 0, 0], 5).sum(), 1)),
      "; ligne 0 de S⁵ :", bool(np.allclose(bikes_after(S_24, [1, 0, 0], 5), np.linalg.matrix_power(S_24, 5)[0])),
      "(attendu True, True)")
pi_check_24 = market_shares(P_OPERATORS)
print("(2) les opérateurs : πP = π :", bool(np.allclose(pi_check_24 @ P_OPERATORS, pi_check_24)), "(attendu True)")
V_check_24 = subscription_value(P_SUBSCRIPTION_24, 12.0, 15.0, 0.96)
r_check_24 = np.array([0.0, 12.0, 15.0])
print("(3) résidu ‖V − (r + γPV)‖∞ < 1e-9 :",
      bool(np.abs(V_check_24 - (r_check_24 + 0.96 * P_SUBSCRIPTION_24 @ V_check_24)).max() < 1e-9), "(attendu True)")
print("(4) le cursus : les durées moyennes :",
      bool(np.allclose(total_takings(Q_CURSUS, np.ones(3)), np.linalg.solve(np.eye(3) - Q_CURSUS, np.ones(3)))),
      "(attendu True)")

In [ ]:
def bikes_after_fixed(P, mu0, n):
    """The law of a bike after n days."""
    raise NotImplementedError("bikes_after_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def market_shares_fixed(P):
    """The stationary law."""
    raise NotImplementedError("market_shares_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def subscription_value_fixed(P, monthly_fee, closing_fee, gamma):
    """The values of a subscription: trial (0), paying (1) who pays monthly_fee each month, cancelled (2, absorbing);
    a cancellation costs closing_fee once to the subscriber."""
    raise NotImplementedError("subscription_value_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def total_takings_fixed(P, r):
    """The mean total takings until the end, from each state; ValueError when the process does not end for sure."""
    raise NotImplementedError("total_takings_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M26_24a = answer_M26_24b = answer_M26_24c = answer_M26_24d = ...
with wbm.attempt("M26.24"):          # your four functions: ⏳ while they are not written
    answer_M26_24a = ...  # a) a list of 4 numbers, as computed: the law of the bikes after 3 days by bikes_after_fixed, then the sum of what the WRONG bikes_after returns
    answer_M26_24b = ...  # b) a list of 4 numbers, as computed: the law of market_shares_fixed (operators), then |the sum| of what the WRONG market_shares returns
    answer_M26_24c = ...  # c) a list of 4 numbers, as computed: the values of subscription_value_fixed (the colleague's model), then the value that the WRONG subscription_value gives to the cancelled state
    answer_M26_24d = ...  # d) True or False: total_takings_fixed raises a ValueError on P_ESTIMATED_24

In [ ]:
for letter, answer in zip("abcd", [answer_M26_24a, answer_M26_24b, answer_M26_24c, answer_M26_24d]):
    check_answer(f"M26.24{letter}", answer)

results_24, parts_24 = [], []


def report_24(ok, success, failure):
    results_24.append(bool(ok))
    print(("✅ " + success) if ok else ("❌ " + failure))


def call_24(part, name, function, *args):
    """function(*args); an error of the learner's code becomes one ❌ line that names the function (the check of this
    part stops there, the final verdict stays)."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            return function(*args)
    except (NotImplementedError, MymathMissingError):
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the function
        report_24(False, "", f"({part}) {name} : ton code lève {type(exc).__name__} : {str(exc)[:200]}")
        parts_24.append(part)
        raise StopCheck from None


rng_c24 = np.random.default_rng(6204)


def chain_c24(m, sparse=0.0):
    """A random stochastic chain of m states (positive entries, a few zeros when sparse > 0)."""
    W = rng_c24.random((m, m)) * (rng_c24.random((m, m)) >= sparse) + 0.02 * np.eye(m)
    return W / W.sum(axis=1, keepdims=True)


with checked("M26.24"):                                       # (1) bikes_after_fixed
    wrong_24 = []
    for m_c24, n_c24 in ((4, 0), (4, 7), (5, 3)):
        P_c24, mu_c24 = chain_c24(m_c24, 0.3), rng_c24.dirichlet(np.ones(m_c24))
        want_c24 = mu_c24 @ np.linalg.matrix_power(P_c24, n_c24)
        got_c24 = call_24(1, "bikes_after_fixed", bikes_after_fixed, P_c24.copy(), mu_c24.copy(), n_c24)
        if matches(got_c24, want_c24):
            continue
        if matches(got_c24, np.linalg.matrix_power(P_c24, n_c24) @ mu_c24):
            wrong_24.append("tu fais encore avancer la loi par P @ mu : une loi est une ligne, mu @ P")
        else:
            wrong_24.append(f"{m_c24} états, {n_c24} jours : attendu {shown(want_c24)}, reçu {shown(got_c24)[:120]}")
    report_24(not wrong_24, "(1) bikes_after_fixed : des chaînes que tu ne vois pas, non symétriques, 0, 3 et 7 jours",
              "(1) bikes_after_fixed : " + " ; ".join(list(dict.fromkeys(wrong_24))[:2]))
    parts_24.append(1)

with checked("M26.24"):                                       # (2) market_shares_fixed
    wrong_24 = []
    for m_c24 in (4, 5, 6):
        P_c24 = chain_c24(m_c24)
        v_c24 = linalg.null_space(P_c24.T - np.eye(m_c24))[:, 0]
        want_c24 = v_c24 / v_c24.sum()
        got_c24 = call_24(2, "market_shares_fixed", market_shares_fixed, P_c24.copy())
        if matches(got_c24, want_c24):
            continue
        try:
            arr_c24 = np.asarray(got_c24, dtype=float)
            proportional_c24 = arr_c24.shape == want_c24.shape and matches(arr_c24 / arr_c24.sum(), want_c24)
        except (TypeError, ValueError, ZeroDivisionError):
            proportional_c24 = False
        if proportional_c24:
            wrong_24.append("ta loi est proportionnelle à la bonne : divise le vecteur propre par sa somme (son signe est "
                            "arbitraire, la division le corrige)")
        else:
            wrong_24.append(f"{m_c24} états : attendu {shown(want_c24)}, reçu {shown(got_c24)[:120]}")
    report_24(not wrong_24, "(2) market_shares_fixed : des chaînes que tu ne vois pas, de 4 à 6 états",
              "(2) market_shares_fixed : " + " ; ".join(list(dict.fromkeys(wrong_24))[:2]))
    parts_24.append(2)

with checked("M26.24"):                                       # (3) subscription_value_fixed
    wrong_24 = []
    for p_paid_c24, p_stay_c24, monthly_c24, closing_c24, gamma_c24 in ((0.7, 0.85, 20.0, 30.0, 0.85),
                                                                         (0.4, 0.95, 8.0, 5.0, 0.96)):
        P_c24 = np.array([[0, p_paid_c24, 1 - p_paid_c24], [0, p_stay_c24, 1 - p_stay_c24], [0, 0, 1.0]])
        r_c24 = np.array([closing_c24 * P_c24[0, 2], monthly_c24 + closing_c24 * P_c24[1, 2], 0.0])
        want_c24 = np.linalg.solve(np.eye(3) - gamma_c24 * P_c24, r_c24)
        got_c24 = call_24(3, "subscription_value_fixed", subscription_value_fixed, P_c24.copy(), monthly_c24, closing_c24,
                          gamma_c24)
        if matches(got_c24, want_c24):
            continue
        got_arr_c24 = np.asarray(got_c24, dtype=float) if matches(got_c24, np.zeros(3), rtol=np.inf, atol=np.inf) else None
        if got_arr_c24 is not None and abs(got_arr_c24[2]) > 1e-9:
            wrong_24.append("l'état résilié rapporte encore : un état absorbant reçoit une récompense nulle, sinon il la "
                            "rapporte à chaque mois, pour toujours")
        elif matches(got_c24, np.linalg.solve(np.eye(3) - gamma_c24 * P_c24, [0.0, monthly_c24, 0.0])):
            wrong_24.append("les frais de résiliation ont disparu : ils se paient une fois, sur la transition vers « résilié »")
        else:
            wrong_24.append(f"un autre abonnement (γ = {fr(gamma_c24, 2)}) : attendu {shown(want_c24)}, reçu {shown(got_c24)}")
    report_24(not wrong_24, "(3) subscription_value_fixed : deux abonnements que tu ne vois pas",
              "(3) subscription_value_fixed : " + " ; ".join(list(dict.fromkeys(wrong_24))[:2]))
    parts_24.append(3)

with checked("M26.24"):                                       # (4) total_takings_fixed
    wrong_24 = []
    ends_c24 = chain_c24(4, 0.2) * np.array([[0.9], [0.8], [0.95], [0.7]])     # every row ends: a sure end
    r_c24 = rng_c24.uniform(0.5, 2.0, 4)
    got_c24 = call_24(4, "total_takings_fixed", total_takings_fixed, ends_c24.copy(), r_c24.copy())
    if not matches(got_c24, np.linalg.solve(np.eye(4) - ends_c24, r_c24)):
        wrong_24.append(f"une chaîne qui finit avec certitude : attendu {shown(np.linalg.solve(np.eye(4) - ends_c24, r_c24))}, "
                        f"reçu {shown(got_c24)}")
    kept_c24 = np.zeros((4, 4))                               # an absorbing end kept as a state: I − P singular
    kept_c24[:3, :3], kept_c24[:3, 3], kept_c24[3, 3] = ends_c24[:3, :3], 1 - ends_c24[:3, :3].sum(axis=1), 1.0
    for label_c24, P_c24 in (("une chaîne stochastique, sans fin", chain_c24(5)),
                             ("une fin gardée comme état absorbant", kept_c24)):
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                total_takings_fixed(P_c24.copy(), np.ones(len(P_c24)))
            wrong_24.append(f"{label_c24} : attendu une ValueError (la fin n'est pas sûre, I − P est singulière ; teste "
                            "np.linalg.matrix_rank), mais rien n'a été levé")
        except (NotImplementedError, MymathMissingError):
            raise
        except np.linalg.LinAlgError:
            wrong_24.append(f"{label_c24} : np.linalg.solve a échoué tout seul ; teste d'abord np.linalg.matrix_rank(I − P) "
                            "et lève toi-même une ValueError")
        except ValueError:
            pass
    report_24(not wrong_24, "(4) total_takings_fixed : une fin sûre résolue, deux fins pas sûres refusées",
              "(4) total_takings_fixed : " + " ; ".join(wrong_24[:2]))
    parts_24.append(4)

if len(set(parts_24)) == 4:
    verdict("M26.24", all(results_24), "les quatre fonctions corrigées passent tous les cas.",
            "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pour chaque erreur, pourquoi la vérification du collègue ne la voyait-elle pas (une matrice symétrique, une propriété qui ne voit pas la normalisation, un résidu qui ne contrôle que la résolution, une chaîne qui finit), et quelle vérification l'aurait vue ? Que rendrait `bellman_linear_solve` pour les vélos avec $\gamma = 0{,}96$ ?

### Ex M26.25 — Un test de propriété : πP = π, et pour `bellman_fixed_point_iterates`, V_k − V tenu par γᵏ‖V₀ − V‖∞ et V = r + γPV pour la limite, sur des chaînes tirées au hasard 🛠️ ★ ⏱️ 20 min
**Objectif :** écrire un test de propriété qui compte ses échecs sur beaucoup de chaînes tirées au hasard, avec une tolérance justifiée, et voir qu'une propriété nécessaire ne suffit pas.  
**Prérequis :** fiche §2 et §6 ; Ex M26.17 à M26.19 · **Parcours :** C

La cellule fournie donne `random_chain_25(rng)`, qui tire une chaîne de 2 à 8 états, plus ou moins creuse, stochastique ou sous-stochastique (une fois sur deux), des récompenses et un $\gamma$ parmi 0,6, 0,7, 0,85 et 0,96, et rend `(P, r, gamma, stochastic)` ; et trois versions fausses : `stationary_times_two` (rend $2\boldsymbol{\pi}$), `iterates_in_place` (des balayages en place au lieu des itérés synchrones) et `iterates_without_gamma` (le pas $\mathbf{r} + \mathbf{P}\mathbf{V}$).

Écris `markov_property_test_25(n_trials, rng, stationary=None, iterates=None, tol=1e-9)` ; quand `stationary` ou `iterates` vaut `None`, le test prend `mymath.markov.stationary_distribution` ou `mymath.markov.bellman_fixed_point_iterates` **au moment de l'appel**. Pour chacun des `n_trials` essais, dans cet ordre : `P, r, gamma, stochastic = random_chain_25(rng)`, puis `V0 = rng.normal(0, 10, size=m)` ; si la chaîne est stochastique, $\boldsymbol{\pi}$ = `stationary(P)` (un essai où elle lève une `ValueError`, une chaîne à plusieurs classes fermées, se saute pour cette partie) ; $\mathbf{V}$ = `mymath.markov.bellman_linear_solve(P, r, gamma)` ; les itérés = `iterates(P, r, gamma, V0, 600)` ; avec `scale = 1 + max|V|`. La fonction rend un dictionnaire de six compteurs d'essais et d'un rapport :
- `"law"` : $\boldsymbol{\pi}$ a une coordonnée inférieure à $-$`tol` ou une somme qui s'écarte de 1 de plus de `tol` ;
- `"invariance"` : $\lVert\boldsymbol{\pi}\mathbf{P} - \boldsymbol{\pi}\rVert_\infty >$ `tol` ;
- `"recurrence"` : une ligne $k + 1$ des itérés s'écarte de $\mathbf{r} + \gamma\mathbf{P}\mathbf{V}_k$ de plus de `tol * scale` ;
- `"a_priori"` : il existe $k$ avec $\lVert\mathbf{V}_k - \mathbf{V}\rVert_\infty > \gamma^k\lVert\mathbf{V}_0 - \mathbf{V}\rVert_\infty + $ `tol * scale` ;
- `"a_posteriori"` : il existe $k \ge 1$ avec $\lVert\mathbf{V}_k - \mathbf{V}\rVert_\infty > \frac{\gamma}{1 - \gamma}\lVert\mathbf{V}_k - \mathbf{V}_{k-1}\rVert_\infty + $ `tol * scale` ;
- `"fixed_point"` : la dernière ligne s'écarte de $\mathbf{r} + \gamma\mathbf{P}\mathbf{V}_{600}$ de plus de `tol * scale` ;
- `"worst_ratio"` : le plus grand rapport $\lVert\mathbf{V}_k - \mathbf{V}\rVert_\infty / (\gamma^k\lVert\mathbf{V}_0 - \mathbf{V}\rVert_\infty)$, sur les essais et les $k$ où le dénominateur dépasse $10^{-6}$ `* scale`.

Puis :
- a) Dans ta copie : justifie `tol = 1e-9` : de quel ordre sont les erreurs d'arrondi sur ces calculs ?
- b) Avec `rng = np.random.default_rng(6225)` et 300 essais, avec tes fonctions : vrai ou faux, les six compteurs sont nuls ; et le pire rapport (une liste : le booléen, puis le nombre ; 4 décimales dans ta copie). Pourquoi est-il si proche de 1 ?
- c) Avec le même départ (un nouveau `default_rng(6225)`, 300 essais) et `stationary=stationary_times_two` : le compteur `"law"`, et vrai ou faux, `"invariance"` reste nul (une liste : l'entier, puis le booléen) ;
- d) avec `iterates=iterates_in_place` : le compteur `"recurrence"`, et deux vrai ou faux : `"a_priori"` reste nul, `"a_posteriori"` reste nul (une liste : l'entier, puis les deux booléens) ;
- e) avec `iterates=iterates_without_gamma` : vrai ou faux, `"a_priori"` compte les 300 essais.

La vérification lance ton test sur deux départs que tu ne vois pas (40 essais chacun) : avec tes fonctions, où ses six compteurs doivent valoir ceux de la référence (0) et son pire rapport le sien ; avec les trois versions fausses, où il doit compter **exactement** les essais de la référence. Elle remplace aussi un instant `mymath.markov.stationary_distribution` par une version qui ne normalise pas, puis `mymath.markov.bellman_fixed_point_iterates` par des itérés qui oublient $\mathbf{r}$ : tes compteurs `"law"`, puis `"recurrence"`, doivent alors compter. Un entier NumPy convient pour un compteur ; un test qui échoue ne lève pas d'`AssertionError` : il compte. Ce qu'il faut retenir : une propriété nécessaire ne suffit pas ($2\boldsymbol{\pi}$ vérifie encore $\boldsymbol{\pi}\mathbf{P} = \boldsymbol{\pi}$ ; un balayage en place vérifie encore les deux bornes) ; un bon test combine des propriétés qui, ensemble, caractérisent le résultat.

In [ ]:
GAMMAS_25 = (0.6, 0.7, 0.85, 0.96)                         # the discount factors of the random chains


def random_chain_25(rng):
    """A random chain of 2 to 8 states, more or less sparse, stochastic or substochastic (one time in two), rewards and
    a γ among GAMMAS_25: returns (P, r, gamma, stochastic)."""
    m = int(rng.integers(2, 9))
    mask = rng.random((m, m)) < rng.uniform(0.3, 1.0)
    mask[np.arange(m), rng.integers(0, m, size=m)] = True     # at least one transition per row
    W = rng.random((m, m)) * mask
    P = W / W.sum(axis=1, keepdims=True)
    stochastic = bool(rng.random() < 0.5)
    if not stochastic:
        P = P * rng.uniform(0.5, 1.0, size=(m, 1))           # rows that may end
    r = rng.normal(0.0, 5.0, size=m)
    gamma = float(rng.choice(GAMMAS_25))
    return P, r, gamma, stochastic


def stationary_times_two(P):
    """WRONG: twice the stationary law (the reference, multiplied by 2)."""
    return 2 * ref_markov.stationary_distribution(P)


def iterates_in_place(P, r, gamma, V0=None, n_steps=10):
    """WRONG: in-place sweeps instead of the synchronous iterates (same arguments, same shape)."""
    P, r = np.asarray(P, dtype=float), np.asarray(r, dtype=float)
    V = np.zeros(len(r)) if V0 is None else np.array(V0, dtype=float)
    out = [V.copy()]
    for _ in range(n_steps):
        for i in range(len(r)):
            V[i] = r[i] + gamma * (P[i] @ V)
        out.append(V.copy())
    return np.array(out)


def iterates_without_gamma(P, r, gamma, V0=None, n_steps=10):
    """WRONG: the step r + P V, without gamma."""
    P, r = np.asarray(P, dtype=float), np.asarray(r, dtype=float)
    V = np.zeros(len(r)) if V0 is None else np.array(V0, dtype=float)
    out = [V.copy()]
    for _ in range(n_steps):
        V = r + P @ V
        out.append(V.copy())
    return np.array(out)

In [ ]:
def markov_property_test_25(n_trials, rng, stationary=None, iterates=None, tol=1e-9):
    """n_trials random chains (random_chain_25, then V0 = rng.normal(0, 10, size=m)); stationary and iterates default
    to mymath.markov.stationary_distribution and bellman_fixed_point_iterates, looked up at the call; V by
    mymath.markov.bellman_linear_solve; 600 iterates; scale = 1 + max|V|. Returns the numbers of trials "law",
    "invariance", "recurrence", "a_priori", "a_posteriori", "fixed_point" that fail, and "worst_ratio"."""
    raise NotImplementedError("markov_property_test_25 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M26_25b = answer_M26_25c = answer_M26_25d = answer_M26_25e = ...
with wbm.attempt("M26.25"):          # your test and your library: ⏳ while they are not written
    answer_M26_25b = ...  # b) a list of 2: True or False: the six counters are 0 (default_rng(6225), 300 trials, your functions); then the worst ratio, as computed
    answer_M26_25c = ...  # c) a list of 2: the counter "law" with stationary=stationary_times_two (an integer), then True or False: "invariance" stays at 0
    answer_M26_25d = ...  # d) a list of 3: the counter "recurrence" with iterates=iterates_in_place (an integer), then True or False: "a_priori" stays at 0, and "a_posteriori" stays at 0
    answer_M26_25e = ...  # e) True or False: with iterates=iterates_without_gamma, "a_priori" counts the 300 trials

In [ ]:
for letter, answer in zip("bcde", [answer_M26_25b, answer_M26_25c, answer_M26_25d, answer_M26_25e]):
    check_answer(f"M26.25{letter}", answer)

with checked("M26.25"):
    results_25 = []
    report_25 = reporter(results_25, stop=False)
    COUNTERS_25 = ("law", "invariance", "recurrence", "a_priori", "a_posteriori", "fixed_point")
    EXPECTED_C25 = {"6241": {"ref": {"law": 0, "invariance": 0, "recurrence": 0, "a_priori": 0, "a_posteriori": 0, "fixed_point": 0, "worst_ratio": 1.0000000000140012}, "times_two": {"law": 23, "invariance": 0, "recurrence": 0, "a_priori": 0, "a_posteriori": 0, "fixed_point": 0, "worst_ratio": 1.0000000000140012}, "in_place": {"law": 0, "invariance": 0, "recurrence": 35, "a_priori": 0, "a_posteriori": 0, "fixed_point": 0, "worst_ratio": 1.0000000000140012}, "without_gamma": {"law": 0, "invariance": 0, "recurrence": 40, "a_priori": 40, "a_posteriori": 40, "fixed_point": 40, "worst_ratio": 11931780.450423412}}, "6242": {"ref": {"law": 0, "invariance": 0, "recurrence": 0, "a_priori": 0, "a_posteriori": 0, "fixed_point": 0, "worst_ratio": 1.0}, "times_two": {"law": 22, "invariance": 0, "recurrence": 0, "a_priori": 0, "a_posteriori": 0, "fixed_point": 0, "worst_ratio": 1.0}, "in_place": {"law": 0, "invariance": 0, "recurrence": 40, "a_priori": 0, "a_posteriori": 0, "fixed_point": 0, "worst_ratio": 1.0}, "without_gamma": {"law": 0, "invariance": 0, "recurrence": 40, "a_priori": 40, "a_posteriori": 40, "fixed_point": 40, "worst_ratio": 11988938.740287418}}}   # the reference, per hidden start

    def run_c25(label, seed, **kwargs):
        """Your markov_property_test_25 on the hidden start seed (40 trials): its dictionary."""
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                with np.errstate(over="ignore", invalid="ignore"):
                    counts = markov_property_test_25(40, np.random.default_rng(seed), **kwargs)
        except (NotImplementedError, MymathMissingError):
            raise
        except AssertionError as exc:
            print(f"❌ Ex M26.25 : markov_property_test_25 ({label}) lève AssertionError ({str(exc)[:120]}) : il doit "
                  "compter les essais qui échouent et rendre le dictionnaire, sans assert.")
            raise StopCheck from None
        except Exception as exc:  # noqa: BLE001 - the learner's error
            print(f"❌ Ex M26.25 : markov_property_test_25 ({label}) lève {type(exc).__name__} : {str(exc)[:200]}")
            raise StopCheck from None
        if not isinstance(counts, dict) or set(counts) != set(COUNTERS_25) | {"worst_ratio"}:
            print("❌ Ex M26.25 : markov_property_test_25 doit rendre un dictionnaire de clés " + ", ".join(COUNTERS_25)
                  + f" et worst_ratio, reçu {shown(counts)[:160]}")
            raise StopCheck
        for key in COUNTERS_25:
            value = counts[key]
            if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)):
                print(f"❌ Ex M26.25 : le compteur \"{key}\" doit être un nombre entier d'essais (un entier NumPy convient), "
                      f"reçu {type(value).__name__} {value!r}")
                raise StopCheck
        try:
            counts["worst_ratio"] = float(counts["worst_ratio"])
        except (TypeError, ValueError):
            print(f"❌ Ex M26.25 : worst_ratio doit être un nombre, reçu {counts['worst_ratio']!r}")
            raise StopCheck from None
        return {key: (int(v) if key != "worst_ratio" else v) for key, v in counts.items()}

    for seed_c25 in (6241, 6242):
        expected_c25 = EXPECTED_C25[str(seed_c25)]
        mine_c25 = run_c25(f"départ caché {seed_c25}", seed_c25)
        wrong_c25 = [k for k in COUNTERS_25 if mine_c25[k] != expected_c25["ref"][k]]
        report_25(not wrong_c25, f"départ caché {seed_c25}, tes fonctions : les six compteurs valent ceux de la référence (0)",
                  f"départ caché {seed_c25}, tes fonctions : " + ", ".join(f"{k} = {mine_c25[k]}" for k in wrong_c25)
                  + " au lieu de 0 : vérifie l'ordre des tirages (random_chain_25, puis V0), tes tolérances (tol, et "
                  "tol * scale pour les valeurs), puis tes fonctions de M26.17 à M26.19 (leurs tests)")
        ratio_c25 = expected_c25["ref"]["worst_ratio"]
        report_25(abs(mine_c25["worst_ratio"] - ratio_c25) <= 1e-6 * max(1.0, ratio_c25),
                  f"départ caché {seed_c25} : le pire rapport vaut celui de la référence ({fr(ratio_c25, 6)})",
                  f"départ caché {seed_c25} : pire rapport {fr(mine_c25['worst_ratio'], 6)}, la référence {fr(ratio_c25, 6)} : "
                  "le rapport ‖V_k − V‖∞ / (γᵏ‖V₀ − V‖∞), sur les essais et les k où le dénominateur dépasse 1e-6 * scale")
        for label_c25, key_c25, kwargs_c25, focus_c25, zeros_c25 in (
                ("stationary_times_two", "times_two", {"stationary": stationary_times_two}, "law", ("invariance",)),
                ("iterates_in_place", "in_place", {"iterates": iterates_in_place}, "recurrence",
                 ("a_priori", "a_posteriori")),
                ("iterates_without_gamma", "without_gamma", {"iterates": iterates_without_gamma}, "a_priori", ())):
            got_c25 = run_c25(f"{label_c25}, départ caché {seed_c25}", seed_c25, **kwargs_c25)
            want_c25 = expected_c25[key_c25]
            report_25(got_c25[focus_c25] == want_c25[focus_c25] and all(got_c25[z] == 0 for z in zeros_c25),
                      f"{label_c25}, départ caché {seed_c25} : \"{focus_c25}\" compte exactement {want_c25[focus_c25]} essais"
                      + (", et " + ", ".join(f"\"{z}\"" for z in zeros_c25)
                         + (" restent nuls" if len(zeros_c25) > 1 else " reste nul") if zeros_c25 else ""),
                      f"{label_c25}, départ caché {seed_c25} : \"{focus_c25}\" = {got_c25[focus_c25]}, la référence "
                      f"{want_c25[focus_c25]}" + "".join(f" ; \"{z}\" = {got_c25[z]}, attendu 0" for z in zeros_c25
                                                         if got_c25[z] != 0)
                      + " : un essai compte une fois s'il échoue, au-delà de la tolérance de l'énoncé")

    def patched_c25(name, replacement, label, key):
        """Your test on the first hidden start while mymath.markov.<name> (and every name of the notebook bound to it) is
        replaced by a wrong function; everything is put back afterwards."""
        original = getattr(mymath.markov, name)
        aliases = [alias for alias, value in list(globals().items()) if value is original]
        calls = [0]

        def wrong(*args, **kwargs):
            calls[0] += 1
            return replacement(*args, **kwargs)

        setattr(mymath.markov, name, wrong)
        for alias in aliases:
            globals()[alias] = wrong
        try:
            counts = run_c25(label, 6241)
        finally:
            setattr(mymath.markov, name, original)
            for alias in aliases:
                globals()[alias] = original
        if calls[0] == 0:
            report_25(False, "", f"{label} remplace un instant ton {name}, mais ton test ne l'appelle jamais : avec "
                      f"{'stationary' if 'stationary' in name else 'iterates'}=None, il doit prendre mymath.markov.{name} au "
                      "moment de l'appel, sans en garder une copie")
        else:
            report_25(counts[key] >= 1, f"{label} à la place de ton {name} : ton compteur \"{key}\" le voit ({counts[key]})",
                      f"{label} à la place de ton {name} : ton compteur \"{key}\" reste à 0")

    def unnormalized_c25(P):
        """WRONG: an eigenvector of Pᵀ for 1, of euclidean norm 1, never divided by its sum."""
        return np.abs(linalg.null_space(np.asarray(P, dtype=float).T - np.eye(len(P)))[:, 0])

    def iterates_without_r_c25(P, r, gamma, V0=None, n_steps=10):
        """WRONG: the iterates of the operator V ↦ γPV (the rewards forgotten)."""
        return ref_markov.bellman_fixed_point_iterates(P, np.zeros(len(r)), gamma, V0, n_steps)

    patched_c25("stationary_distribution", unnormalized_c25, "une loi stationnaire qui ne se normalise pas", "law")
    patched_c25("bellman_fixed_point_iterates", iterates_without_r_c25, "des itérés qui oublient r", "recurrence")
    verdict("M26.25", all(results_25), "ton test compte juste sur la référence et sur les versions fausses, et prend tes "
            "fonctions au moment de l'appel.", "relis les lignes ❌ ci-dessus.")

**Dans ta copie** : a) la justification de `tol = 1e-9`.

**Après la vérification**, dans ta copie : pourquoi `stationary_times_two` passe-t-il le contrôle d'invariance, et `iterates_in_place` les deux bornes ? Quelles propriétés, prises ensemble, caractérisent la loi stationnaire, puis les itérés synchrones ?

## Partie D · Défi

Fiche §4 et §8. La valeur d'une page d'un petit web, estimée par l'arrêt géométrique sous un budget de pas simulés, et jugée sur la valeur exacte.

### Ex M26.26 — Défi : estimer la valeur d'un état à 0,01 près en simulant l'arrêt géométrique (continuer avec la probabilité γ) et en sommant les récompenses sans actualisation, sous un budget de pas simulés 🏆 ★★★ ⏱️ 30 min
**Objectif :** estimer une valeur actualisée par une simulation sans biais, l'arrêt géométrique, et dimensionner la simulation par son écart-type et un budget.  
**Prérequis :** fiche §4 et §8 ; M02b, M02c ; Ex M26.5, M26.18 et M26.20 · **Parcours :** C

Le petit web de M26.20 ($\mathbf{P}_d$, $d = 0{,}85$ : `P_WEB_26`) rapporte une recette publicitaire par page vue : 0,40 €, 1,20 €, 0,20 €, 0,80 €, 0 € et 1,60 € pour les pages 1 à 6 (`REVENUE_26`). Après chaque page, le visiteur continue avec la probabilité $\gamma = 0{,}7$ et quitte le site avec 0,3. D'après la fiche (§4), la recette totale moyenne d'une visite qui commence en page 1, sans actualisation, est la valeur $V$ de la page 1 pour $\gamma = 0{,}7$.

Écris `estimate_value_26(P, r, gamma, s0, budget, rng)`, qui simule des visites depuis l'état `s0` : à chaque page vue, la visite encaisse `r` de cette page, puis continue avec la probabilité `gamma` (sinon elle s'arrête) et passe à la page suivante selon la ligne de `P` ; elle somme les recettes **sans actualisation**. Chaque page vue est un pas simulé ; le nombre total de pas, toutes visites comprises, ne doit jamais dépasser `budget`, et toute visite simulée est menée à son terme et comptée (choisis donc le nombre de visites avec une marge). La fonction rend `(estimate, n_visits, n_steps)` : la moyenne des recettes totales, le nombre de visites et le nombre de pas simulés.

**Critère de réussite :** avec `budget = 600_000` (`BUDGET_26`), pour chacune des dix graines 6230 à 6239 (`SEEDS_26`, `rng = np.random.default_rng(seed)`), `n_steps <= budget`, et l'estimation tombe à moins de 0,01 de la valeur exacte (par `bellman_linear_solve`) au moins 8 fois sur 10. La vérification compare ensuite la moyenne de tes dix estimations à la valeur exacte du modèle (à moins de 4 erreurs types, avec tes nombres de visites) ; elle contrôle, sur une chaîne cachée où chaque page rapporte 1 avec $\gamma = 0{,}96$, que `estimate * n_visits` égale `n_steps` (chaque pas compté, aucune actualisation ajoutée) et que l'estimation est à moins de 4 erreurs types de $\frac{1}{1 - 0{,}96} = 25$ (l'arrêt géométrique, sans troncature) ; et, sur une autre chaîne cachée avec $\gamma = 0{,}6$, que l'estimation est à moins de 4 erreurs types de la valeur exacte. Relève :
- a) la valeur exacte $V$ de la page 1, par ta `bellman_linear_solve` (un nombre ; 4 décimales dans ta copie) ;
- b) l'écart-type de la recette totale d'une visite, estimé sur une simulation pilote de 20 000 visites (la graine que tu veux), et le nombre de visites qu'il faudrait pour une erreur type de 0,004 (une liste de 2 nombres ; 2 décimales et un entier dans ta copie) : la vérification l'accepte à 3 % près de l'écart-type exact du modèle ;
- c) dans ta copie : tes dix estimations et ton nombre de succès (affichés par la vérification).

Puis estime de même l'écart-type du retour actualisé $\sum_{t < 60}\gamma^tR_t$ d'une visite qui ne s'arrête pas : lequel des deux écarts-types est le plus grand, et que coûte la troncature à 60 pas ? Ce qu'il faut retenir : l'arrêt géométrique donne une estimation sans biais de la valeur actualisée, au prix d'une variance plus grande.

In [ ]:
LINKS_26 = {1: [2, 3], 2: [4], 3: [1, 4, 5], 4: [1, 3], 5: [4, 6], 6: []}   # the small web of M26.20
L_26 = np.zeros((6, 6))
for page_26, targets_26 in LINKS_26.items():
    L_26[page_26 - 1] = np.full(6, 1 / 6) if not targets_26 else np.isin(np.arange(1, 7), targets_26) / len(targets_26)
P_WEB_26 = 0.85 * L_26 + 0.15 / 6                          # P_d, d = 0.85 (built here: data never depend on your code)
REVENUE_26 = np.array([0.40, 1.20, 0.20, 0.80, 0.0, 1.60]) # € per page viewed, pages 1 to 6
GAMMA_26 = 0.7                                             # after each page, the visitor goes on with the probability 0.7
BUDGET_26 = 600_000                                        # the budget of simulated steps (pages viewed), per estimate
SEEDS_26 = range(6230, 6240)                               # the ten seeds of the criterion

In [ ]:
def estimate_value_26(P, r, gamma, s0, budget, rng):
    """Visits from the state s0: at each page viewed, collect r of the page, then go on with the probability gamma (else
    stop) to the next page drawn from the row of P; sum the rewards WITHOUT discounting. Each page viewed is one step;
    the total number of steps never exceeds budget, and every visit simulated is run to its end and counted. Returns
    (estimate, n_visits, n_steps): the mean of the totals, the number of visits and the number of steps."""
    raise NotImplementedError("estimate_value_26 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M26_26a = answer_M26_26b = ...
with wbm.attempt("M26.26"):          # your bellman_linear_solve and your simulation: ⏳ while they are not written
    answer_M26_26a = ...  # a) a number, as computed: the exact value V of page 1 (your bellman_linear_solve)
    answer_M26_26b = ...  # b) a list of 2: the standard deviation of the total of a visit (a pilot of 20 000 visits, as computed), then the number of visits for a standard error of 0.004

In [ ]:
for letter, answer in zip("a", [answer_M26_26a]):
    check_answer(f"M26.26{letter}", answer)

def total_sd_c26(P, r, gamma, s0):
    """The exact standard deviation of the undiscounted total of a visit stopped geometrically (continue with γ):
    T = r(S0) + B T′, B ~ Bernoulli(γ), so E[T²] = r² + 2γ r (PV) + γ P E[T²]."""
    P, r = np.asarray(P, dtype=float), np.asarray(r, dtype=float)
    V = np.linalg.solve(np.eye(len(r)) - gamma * P, r)
    second = np.linalg.solve(np.eye(len(r)) - gamma * P, r ** 2 + 2 * gamma * r * (P @ V))
    return float(math.sqrt(second[s0] - V[s0] ** 2)), float(V[s0])


def triple_c26(result, budget):
    """(estimate, n_visits, n_steps) as a float and two integers, n_steps within the budget; else ❌ and StopCheck."""
    estimate, n_visits, n_steps = pair("M26.26", "estimate_value_26", result, names=("estimate", "n_visits", "n_steps"))
    try:
        estimate = float(estimate)
    except (TypeError, ValueError):
        verdict("M26.26", False, "", f"estimate doit être un nombre, reçu {estimate!r}")
        raise StopCheck from None
    for name, value in (("n_visits", n_visits), ("n_steps", n_steps)):
        if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)):
            verdict("M26.26", False, "", f"{name} doit être un entier, reçu {value!r}")
            raise StopCheck
    if n_steps > budget:
        verdict("M26.26", False, "", f"budget dépassé : n_steps = {int(n_steps)} > {budget} : choisis le nombre de visites avec "
                "une marge (une visite dure en moyenne 1/(1 − γ) pages), et mène chaque visite à son terme")
        raise StopCheck
    return estimate, int(n_visits), int(n_steps)


with checked("M26.26"):
    sd_c26, exact_c26 = total_sd_c26(P_WEB_26, REVENUE_26, GAMMA_26, 0)
    runs_c26 = []
    for seed_c26 in SEEDS_26:
        with own_code("M26.26", "estimate_value_26", REF_MARKOV):
            result_c26 = estimate_value_26(P_WEB_26.copy(), REVENUE_26.copy(), GAMMA_26, 0, BUDGET_26,
                                           np.random.default_rng(seed_c26))
        runs_c26.append(triple_c26(result_c26, BUDGET_26))
    estimates_c26 = np.array([run[0] for run in runs_c26])
    hits_c26 = int(np.sum(np.abs(estimates_c26 - exact_c26) < 0.01))
    print("tes dix estimations :", " ; ".join(fr(e, 4) for e in estimates_c26), f"(valeur exacte {fr(exact_c26, 4)})")
    print("visites :", [run[1] for run in runs_c26][:3], "… ; pas simulés :", [run[2] for run in runs_c26][:3], "…")
    verdict("M26.26", hits_c26 >= 8, f"critère atteint : {hits_c26} estimations sur 10 à moins de 0,01, sans dépasser le "
            "budget.", f"critère pas encore atteint : {hits_c26} estimations sur 10 à moins de 0,01 (il en faut 8).")
    se_mean_c26 = math.sqrt(sum(sd_c26 ** 2 / run[1] for run in runs_c26)) / len(runs_c26)
    gap_c26 = float(estimates_c26.mean() - exact_c26)
    verdict("M26.26", abs(gap_c26) <= 4 * se_mean_c26,
            f"la moyenne de tes dix estimations est à {fr(abs(gap_c26), 4)} de la valeur exacte du modèle, moins de 4 erreurs "
            f"types ({fr(4 * se_mean_c26, 4)}) : pas de biais visible.",
            f"la moyenne de tes dix estimations s'écarte de {fr(gap_c26, 4)} de la valeur exacte du modèle, plus de 4 erreurs "
            f"types ({fr(4 * se_mean_c26, 4)}) : ton estimateur a un biais (relis les deux cas cachés ci-dessous).")

with checked("M26.26"):
    hidden_c26 = np.array([[0.1, 0.5, 0.2, 0.2, 0.0], [0.3, 0.1, 0.3, 0.0, 0.3], [0.0, 0.4, 0.2, 0.2, 0.2],
                           [0.25, 0.25, 0.0, 0.25, 0.25], [0.2, 0.0, 0.5, 0.1, 0.2]])
    gamma_a_c26 = 0.96                                        # a hidden chain where every page brings 1
    with own_code("M26.26", "estimate_value_26", REF_MARKOV):
        result_a_c26 = estimate_value_26(hidden_c26.copy(), np.ones(5), gamma_a_c26, 2, BUDGET_26,
                                         np.random.default_rng(6243))
    est_a_c26, visits_a_c26, steps_a_c26 = triple_c26(result_a_c26, BUDGET_26)
    sd_a_c26, mean_a_c26 = total_sd_c26(hidden_c26, np.ones(5), gamma_a_c26, 2)          # 25 and √γ/(1 − γ)
    se_a_c26 = sd_a_c26 / math.sqrt(max(visits_a_c26, 1))
    counted_c26 = abs(est_a_c26 * visits_a_c26 - steps_a_c26) <= 1e-6 * max(steps_a_c26, 1)
    if abs(est_a_c26 - 1 / (1 - gamma_a_c26 ** 2)) <= 4 * se_a_c26 + 0.3:
        why_c26 = ("tu actualises en plus de l'arrêt géométrique : l'arrêt avec la probabilité 1 − γ tient déjà lieu "
                   "d'actualisation (fiche §4) ; somme les recettes sans γᵗ")
    elif abs(est_a_c26 - 1 / gamma_a_c26) <= 0.2:
        why_c26 = "l'arrêt est mal tiré : tu continues avec la probabilité 1 − γ ; la visite continue avec la probabilité γ"
    elif abs(est_a_c26 - gamma_a_c26 / (1 - gamma_a_c26)) <= 4 * se_a_c26 and not abs(est_a_c26 - mean_a_c26) <= 4 * se_a_c26:
        why_c26 = ("l'arrêt est mal tiré : la page où la visite commence est toujours vue et encaissée ; le tirage "
                   "« continuer ou s'arrêter » vient après chaque page")
    elif not counted_c26:
        why_c26 = ("estimate × n_visits ne vaut pas n_steps alors que chaque page rapporte 1 : chaque page vue est un pas, "
                   "compté une fois, et la recette d'une visite est la somme de ses pages, sans actualisation")
    elif est_a_c26 < mean_a_c26 - 4 * se_a_c26:
        why_c26 = "tes visites sont coupées (une troncature ?) : mène chaque visite à son terme, jusqu'au tirage qui l'arrête"
    elif abs(est_a_c26 - mean_a_c26) > 4 * se_a_c26:
        why_c26 = f"l'estimation {fr(est_a_c26, 3)} s'écarte de 25 de plus de 4 erreurs types ({fr(4 * se_a_c26, 3)})"
    else:
        why_c26 = ""
    verdict("M26.26", not why_c26, f"une chaîne cachée où chaque page rapporte 1 (γ = 0,96) : {fr(est_a_c26, 3)}, à moins de 4 "
            f"erreurs types de 1/(1 − γ) = 25, et estimate × n_visits = n_steps ({steps_a_c26} pas).",
            f"une chaîne cachée où chaque page rapporte 1 (γ = 0,96, attendu 25) : {why_c26}.")
    rewards_b_c26 = np.array([2.0, -1.0, 0.5, 3.0, 1.0])
    with own_code("M26.26", "estimate_value_26", REF_MARKOV):
        result_b_c26 = estimate_value_26(hidden_c26.copy(), rewards_b_c26.copy(), 0.6, 4, BUDGET_26,
                                         np.random.default_rng(6245))
    est_b_c26, visits_b_c26, _ = triple_c26(result_b_c26, BUDGET_26)
    sd_b_c26, mean_b_c26 = total_sd_c26(hidden_c26, rewards_b_c26, 0.6, 4)
    se_b_c26 = sd_b_c26 / math.sqrt(max(visits_b_c26, 1))
    verdict("M26.26", abs(est_b_c26 - mean_b_c26) <= 4 * se_b_c26,
            f"une autre chaîne cachée (γ = 0,6, des recettes négatives) : {fr(est_b_c26, 4)}, à moins de 4 erreurs types de "
            "la valeur exacte.", f"une autre chaîne cachée (γ = 0,6, des recettes négatives) : {fr(est_b_c26, 4)}, à plus "
            f"de 4 erreurs types ({fr(4 * se_b_c26, 4)}) de la valeur exacte {fr(mean_b_c26, 4)}.")

with checked("M26.26"):
    if not filled(answer_M26_26b):
        print("⏳ Ex M26.26b : pas encore fait.")
        raise StopCheck
    try:
        pilot_sd_c26, visits_needed_c26 = (float(v) for v in answer_M26_26b)
    except (TypeError, ValueError):
        print("❌ Ex M26.26b : attendu une liste de 2 nombres : l'écart-type de la simulation pilote, puis le nombre de "
              "visites.")
        raise StopCheck from None
    if abs(pilot_sd_c26 - sd_c26) <= 0.03 * sd_c26:
        print(f"✅ Ex M26.26b : écart-type pilote {fr(pilot_sd_c26, 3)}, à moins de 3 % de l'écart-type exact du modèle.")
    elif abs(pilot_sd_c26 - sd_c26 ** 2) <= 0.06 * sd_c26 ** 2:
        print("❌ Ex M26.26b : c'est une variance : b) demande l'écart-type (np.std), sa racine.")
    elif pilot_sd_c26 < 0.5 * sd_c26:
        print("❌ Ex M26.26b : ton écart-type est bien trop petit : b) demande celui de la recette totale d'une visite, "
              "sans actualisation, arrêtée par le tirage de γ (pas celui du retour actualisé, ni celui de la moyenne).")
    else:
        print(f"❌ Ex M26.26b : écart-type pilote {fr(pilot_sd_c26, 3)} : attendu, à 3 % près, celui de la recette totale "
              "d'une visite, depuis la page 1, sur 20 000 visites simulées comme dans estimate_value_26.")
    if abs(visits_needed_c26 - (pilot_sd_c26 / 0.004) ** 2) <= 1 + 1e-9 * visits_needed_c26:
        print("✅ Ex M26.26b : le nombre de visites pour une erreur type de 0,004 est cohérent avec ton écart-type.")
    elif abs(visits_needed_c26 - pilot_sd_c26 / 0.004) <= 1:
        print("❌ Ex M26.26b : l'erreur type vaut σ/√n : n = (σ/0,004)², le carré.")
    else:
        print("❌ Ex M26.26b : le nombre de visites n vérifie σ/√n = 0,004 avec ton écart-type σ : n = (σ/0,004)² (arrondi "
              "vers le haut).")

**Dans ta copie** : c) tes dix estimations et ton nombre de succès ; l'écart-type du retour actualisé tronqué à 60 pages, comparé à celui de b), et le biais de la troncature.

**Pour réfléchir** : pourquoi la recette totale d'une visite arrêtée au hasard varie-t-elle plus que le retour actualisé d'une visite sans fin ? Avec le même budget de pas, laquelle des deux méthodes choisirais-tu pour estimer $V$ à 0,001 près, et que faudrait-il accepter en échange ?

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu écrire une matrice de transition, faire avancer une loi (une ligne) et moyenner une fonction (une colonne) ?
2. Sais-tu calculer une loi stationnaire par un système linéaire, dire si elle est unique et si la loi de la chaîne y converge, et à quelle vitesse ?
3. Sais-tu écrire et résoudre le système de Bellman d'une politique fixée, avec une fin possible, et dire quand $\gamma = 1$ est permis ?
4. Sais-tu itérer l'opérateur de Bellman, borner l'erreur a priori et a posteriori, et choisir le nombre d'itérations ?
5. Sais-tu estimer une valeur par simulation sans biais, et reconnaître le biais du maximum ?

**La suite** : M27, sur les jeux à deux joueurs et les distances entre lois, puis le chapitre 26 de workbookIA, qui applique ces outils à l'apprentissage par renforcement.